# Fine-tune Qwen3.5-4B (LoRA / QLoRA) for LifeMate on Google Colab

Orchestrates the repo `Le-Minh-Nhut/zaloai` — the training loop, collator, checkpoint format,
evaluation and export all come from the package. This notebook only decides *where* things live,
*how much* memory to use, and *when* to sync to Drive.

**Before you click anything:** `Runtime -> Change runtime type -> GPU`. A CPU runtime cannot run
this notebook and still bills Compute Units (CU) while it sits idle.

| section | what it costs | CU (approx., ESTIMATE) |
| --- | --- | --- |
| 1-3 setup, install, clone | 2-4 min, no model | 0.1 |
| 4-5 Drive + dataset build | 3-8 min CPU | 0.2 |
| 6-7 model load + 1-step probe | 5-15 min GPU | T4 0.3 / L4 0.8 / A100 1.5 |
| 8 training (3 epochs, 4368 records) | hours — see the cell's own estimate | the expensive part |
| 9-13 eval, export, smoke test, cleanup | 10-30 min | 0.3-1.0 |

Rates are Colab's published-ish CU/hour for a T4 / L4 / A100 class GPU; treat every number here as
an **ESTIMATE**. Nothing in this notebook was executed on a Colab runtime when it was written.

Ground rules baked into the cells below:

* the model id is passed explicitly (`Qwen/Qwen3.5-4B`); the repo default may change to
  `openbmb/MiniCPM5-2B`, and this notebook must never silently follow it;
* the test splits are never read, never trained on, never scored. Only `train` is used, with the
  repo's own train/val split (coverage: `docs/COMPETITION_COMPLIANCE.md`);
* any mock-environment number is printed with `MOCK ENV / SYNTHETIC` next to it;
* checkpoints are written by the repo atomically, and synced to Drive through a staging directory
  plus a completion marker, so a half-copied directory can never pass as resumable.


## 1. Environment setup — GPU, precision, dependencies, disk

Purpose: know exactly which GPU we got, whether it can do bf16 (**T4 cannot** — its sm_75 has no
bf16 kernels, so the notebook selects fp16 there and bf16 everywhere else), install only the
dependencies that are genuinely missing, and refuse to continue on a runtime that cannot hold a 4B
model. Cost: seconds of compute plus ~1-3 min of pip on a cold runtime (0.1-0.3 CU).

The repo's own estimate for *its default 2B model* is roughly 11-14 GB at `max_seq_length=8192`; for
the 4B model this notebook trains, section 2b derives the number per tier (~17.3 GB for 16-bit LoRA
at 8192 — **ESTIMATE**). The floor here is 14 GB of VRAM — T4 16 GB, L4 24 GB, A100 40 GB, A100
80 GB. Anything smaller stops with an actionable error instead of an OOM after forty minutes of
training.

**fp16 warning, printed by this section whenever the GPU has no bf16 (T4).** fp16 training needs
dynamic loss scaling; the repo trainer supplies it with `torch.amp.GradScaler`
(`src/lifemate/training/trainer.py::_make_grad_scaler`), and that fp16+scaler path is **not verified
on a T4 here** — the scaler backs off and skips a step when it sees inf/NaN, so expect occasional
skipped steps. On a T4 the honest default is `TRAINING_METHOD="qlora"` (section 2b derives it).

This section also exports `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`. Long sequences
fragment the caching allocator; expandable segments let it reuse freed blocks instead of stranding
them. It is set in `os.environ` **before** any child process starts, because the variable is read
when a process creates its CUDA context — so `scripts/train.py` inherits it.


In [ ]:
# --- 1a. Probe the runtime: Colab?, GPU?, bf16?, disk -------------------------
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

try:
    import torch
except ImportError:  # a fresh non-Colab box may have no torch yet
    torch = None

GPU = {"available": False, "reason": "torch is not importable"}
PRECISION = "float16"
if torch is not None and torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    _free, _total = torch.cuda.mem_get_info(0)
    GPU = {
        "available": True,
        "name": str(_props.name),
        "capability": f"{_props.major}.{_props.minor}",
        "vram_total_gb": round(_total / 1e9, 2),
        "vram_free_gb": round(_free / 1e9, 2),
        "bf16_supported": bool(torch.cuda.is_bf16_supported()),
        "cuda": torch.version.cuda,
        "torch": torch.__version__,
        "n_gpus": torch.cuda.device_count(),
    }
    # T4 (sm_75) has NO bf16: asking for it yields NaNs or a silent fp32 upcast.
    PRECISION = "bfloat16" if GPU["bf16_supported"] else "float16"

# Long sequences fragment the caching allocator; expandable_segments lets it reuse freed blocks
# instead of stranding them. Exported here, not inside the trainer: the variable is read when a
# process creates its CUDA context, so it must be in the environment BEFORE the child starts.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if PRECISION == "float16":
    print(
        "\n!! PRECISION=float16: this GPU reports no bf16 support (T4 is sm_75), so every autocast\n"
        "   region runs in fp16. fp16 needs dynamic loss scaling, which the repo trainer provides\n"
        "   via torch.amp.GradScaler (src/lifemate/training/trainer.py::_make_grad_scaler); when the\n"
        "   scaler detects inf/NaN it backs off and SKIPS that optimizer step. That fp16+scaler path\n"
        "   is NOT verified on a T4 here. On a T4 prefer TRAINING_METHOD=\"qlora\" in section 2: the\n"
        "   4-bit base frees ~6 GB of frozen weights, and that is what lets 8192 tokens fit at all.\n"
    )
print(f"allocator      : PYTORCH_CUDA_ALLOC_CONF={os.environ['PYTORCH_CUDA_ALLOC_CONF']} "
      f"(inherited by scripts/train.py)")

_root = "/content" if IN_COLAB else str(Path.home())
_disk = shutil.disk_usage(_root)
print(json.dumps(
    {
        "colab": IN_COLAB,
        "python": sys.version.split()[0],
        "gpu": GPU,
        "chosen_precision": PRECISION,
        "disk_free_gb": round(_disk.free / 1e9, 2),
        "disk_total_gb": round(_disk.total / 1e9, 2),
    },
    indent=2,
))


In [ ]:
# --- 1b. GPU suitability gate + readiness micro-test -------------------------
MIN_VRAM_GB = 14.0
problems = []

if not IN_COLAB:
    print("note: not running on Colab; continuing on this host's GPU if present")
if not GPU.get("available"):
    problems.append(
        "no CUDA GPU visible. Runtime -> Change runtime type -> GPU (T4/L4/A100), "
        "then re-run this cell. A CPU runtime cannot train a 4B model."
    )
else:
    if GPU["vram_total_gb"] < MIN_VRAM_GB:
        problems.append(
            f"{GPU['name']} exposes {GPU['vram_total_gb']} GB of VRAM, below the {MIN_VRAM_GB} GB "
            "floor for this notebook's 4B run. Use a T4/L4/A100 runtime (a 16 GB T4 is the minimum, "
            "and only with the qlora method section 2b derives)."
        )
    if float(GPU["capability"]) < 7.0:
        problems.append(
            f"compute capability {GPU['capability']} is below 7.0 (Volta); bf16/fp16 autocast "
            "kernels are unavailable. Use a T4 or newer."
        )
if _disk.free / 1e9 < 40:
    problems.append(
        f"only {round(_disk.free / 1e9, 1)} GB free on {_root}. Weights (~8.7 GB), the HF cache, "
        "checkpoints and logs need ~40 GB. Free space or mount Drive for checkpoints first."
    )
if problems:
    raise RuntimeError("Runtime is not suitable for this notebook:\n - " + "\n - ".join(problems))

GPU_TIER = "unknown"
if GPU["available"]:
    _gb = GPU["vram_total_gb"]
    GPU_TIER = (
        "A100-80GB" if _gb >= 70 else
        "A100-40GB" if _gb >= 35 else
        "L4-24GB" if _gb >= 22 else
        "T4-16GB"
    )
    # Prove the kernels we are about to rely on actually run on this device.
    _dt = torch.bfloat16 if PRECISION == "bfloat16" else torch.float16
    _a = torch.randn(64, 64, device="cuda", dtype=_dt)
    _c = (_a @ _a).float().sum().item()
    torch.cuda.synchronize()
    del _a
    torch.cuda.empty_cache()
    print(f"GPU tier      : {GPU_TIER} ({GPU['name']}, {GPU['vram_total_gb']} GB, sm_{GPU['capability'].replace('.', '')})")
    print(f"precision     : {PRECISION} (bf16_supported={GPU['bf16_supported']})")
    print(f"readiness test: 64x64 matmul on device OK (checksum {_c:.2f})")
else:
    print("GPU tier      : none")


In [ ]:
# --- 1c. Install only MISSING dependencies ---------------------------------
# Colab ships torch + a few ML packages but not the full training stack, and the versions drift.
# Nothing here is installed unless it is actually absent, so a warm runtime costs one import each.
import importlib.metadata
import importlib.util


def has_module(spec):
    """True when the distribution named by a pip spec is importable."""
    name = re.split(r"[\[<>=!~ ]", spec, maxsplit=1)[0].strip().replace("-", "_")
    return importlib.util.find_spec(name) is not None


def pip_install(*specs):
    print("installing:", ", ".join(specs), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *specs], check=True)


# The training stack. bitsandbytes (QLoRA only) is installed in 3b, once section 2 knows the method.
REQUIRED = ["transformers>=4.44", "peft>=0.11", "accelerate>=0.33", "pyyaml>=6"]

missing = [spec for spec in REQUIRED if not has_module(spec)]
if missing:
    pip_install(*missing)
else:
    print("all required packages present:", ", ".join(REQUIRED))

for _spec in REQUIRED:
    _name = re.split(r"[\[<>=!~ ]", _spec, maxsplit=1)[0]
    try:
        print(f"  {_name:<16} {importlib.metadata.version(_name)}")
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError(
            f"{_name} is still not importable after installation. On Colab this usually means the "
            "install needs a runtime restart; do Runtime -> Restart session and run again from "
            "section 1. Nothing was trained."
        ) from None


## 2. Configuration — one cell, Colab `#@param` forms

Purpose: every knob that changes the run lives here, in one place, so a resumed run can be proven
to use the same numbers. `MAX_SEQ_LENGTH = 0` means "derive it from the measured token distribution"
(section 5 fills it in). Cost: none.

Rules the derived defaults follow (all **ESTIMATE**; nothing below was measured on Colab):

* `TRAINING_METHOD=lora` keeps the frozen base in bf16/fp16 and trains adapters — the repo's
  architecture-aware discovery picks the projections (including Qwen's hybrid linear-attention
  blocks), never a hardcoded Llama list;
* `qlora` puts the *base* in 4-bit NF4 so a T4 has room for a long context; the trained artifact
  is still a normal LoRA adapter;
* effective batch stays 8 sequences (`micro_batch * grad_accum`), which is what the shipped configs
  assume;
* T4 gets the smallest micro-batch because it has no bf16 and the least memory.

**The GPU tier decides the method, not only the sequence length** (cell 2b, all numbers **ESTIMATE**).
Qwen3.5-4B is a top-level `Qwen3_5ForConditionalGeneration` whose text side is ~4B parameters
(hidden 2560, 32 layers). Frozen fp16 weights are therefore ~8.8 GB, and at 8192 tokens the
activation + logits terms add ~7.9 GB — 17.3 GB, against ~14.9 GB usable on a T4 16 GB. So plain
`lora` at that length on `T4-16GB` is derived to `qlora` (10.9 GB) with the arithmetic printed, and
plain `lora` is capped at 4096 there (13.3 GB, tight). L4 keeps `lora` at 8192 and drops to `qlora`
at 12288; A100 40/80 keep `lora` up to 16384/32768.

**The requested `MAX_SEQ_LENGTH` is never lowered silently.** When it cannot fit the tier, cell 2b
prints `!! MAX_SEQ_LENGTH <old> -> <new>` with the estimate that forced the clamp and the way to
keep the old value (a smaller sequence, or a bigger runtime). `MAX_SEQ_LENGTH = 0` means "take the
tier's recommendation".


In [ ]:
#@title 2. Run configuration { display-mode: "form" }
#@markdown Model + tuning
MODEL_ID = "Qwen/Qwen3.5-4B" #@param {type:"string"}
TRAINING_METHOD = "lora" #@param ["lora", "qlora"]
MAX_SEQ_LENGTH = 8192 #@param {type:"integer"}
MICRO_BATCH_SIZE = 1 #@param {type:"integer"}
GRADIENT_ACCUMULATION_STEPS = 8 #@param {type:"integer"}
NUM_EPOCHS = 3 #@param {type:"number"}
LEARNING_RATE = 0.0002 #@param {type:"number"}
#@markdown Storage + resume
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
RESUME_FROM_CHECKPOINT = "" #@param {type:"string"}
MAX_STEPS = 0 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
#@markdown Repo pin and run identity (not part of the required list, but needed downstream)
BRANCH_OR_COMMIT = "main" #@param {type:"string"}
RUN_NAME = "qwen35-4b-lora" #@param {type:"string"}
DATASET_MODE = "auto" #@param ["auto", "rebuild"]

if int(MAX_SEQ_LENGTH) != 0 and not (1024 <= int(MAX_SEQ_LENGTH) <= 32768):
    raise ValueError(f"MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} outside the supported 1024..32768 range")
if TRAINING_METHOD not in ("lora", "qlora"):
    raise ValueError(f"TRAINING_METHOD must be lora or qlora, got {TRAINING_METHOD!r}")
if int(MICRO_BATCH_SIZE) < 1 or int(GRADIENT_ACCUMULATION_STEPS) < 1:
    raise ValueError("MICRO_BATCH_SIZE and GRADIENT_ACCUMULATION_STEPS must be >= 1")
print(
    f"model={MODEL_ID} method={TRAINING_METHOD} (requested; 2b may override) seq={MAX_SEQ_LENGTH or 'auto'} "
    f"micro_batch={MICRO_BATCH_SIZE} accum={GRADIENT_ACCUMULATION_STEPS} "
    f"epochs={NUM_EPOCHS} lr={LEARNING_RATE} max_steps={MAX_STEPS or 'full'} seed={SEED}"
)


In [ ]:
# --- 2b. Derivation: the GPU TIER decides the METHOD, not only the sequence length ----
# Architecture facts behind every number below — Qwen3.5-4B is a top-level
# Qwen3_5ForConditionalGeneration (text side ~4B params, hidden 2560, 32 layers) plus a vision tower.
BASE_PARAMS_B = 4.4
HIDDEN = 2560
LAYERS = 32
VOCAB = 151936

TIER_VRAM_GB = {"T4-16GB": 16.0, "L4-24GB": 24.0, "A100-40GB": 40.0, "A100-80GB": 80.0, "unknown": 16.0}
USABLE_FRACTION = 0.93        # driver + CUDA context + fragmentation reserve
QLORA_BYTES_PER_PARAM = 0.55  # NF4 plus its absmax/quantile doubles
ACT_FACTOR = 4.0              # activation copies kept per layer boundary under gradient checkpointing
RESERVE_GB = 0.6              # allocator slack, cuBLAS workspaces
SEQ_CEILING = 32768

# Hard method ceilings per tier. The arithmetic below already refuses 16-bit LoRA at 8192 on a T4
# (17.3 GB against 14.9 GB usable); these caps also cover what the arithmetic cannot see, such as
# fragmentation growth over a long run and a second process holding a slice of the card.
TIER_METHOD_CAP = {
    "T4-16GB":   {"lora": 4096, "qlora": 8192},
    "L4-24GB":   {"lora": 8192, "qlora": 12288},
    "A100-40GB": {"lora": 16384, "qlora": 16384},
    "A100-80GB": {"lora": 32768, "qlora": 32768},
    "unknown":   {"lora": 4096, "qlora": 8192},
}

def next_multiple(value, multiple=256):
    """Round ``value`` up to the next ``multiple`` (never below one multiple)."""
    return max(multiple, -(-int(value) // multiple) * multiple)

def estimate_vram_gb(gpu_tier, method, seq_len, micro_batch=1, gradient_checkpointing=True):
    """Projected VRAM for one training step. ESTIMATE — a heuristic, never a measurement."""
    weights = BASE_PARAMS_B * (QLORA_BYTES_PER_PARAM if method == "qlora" else 2.0)
    acts = (LAYERS * seq_len * HIDDEN * 2 * int(micro_batch)
            * (ACT_FACTOR if gradient_checkpointing else ACT_FACTOR * 4) / 1e9)
    logits = seq_len * VOCAB * 2 / 1e9          # full-sequence logits; the trainer chunks this term
    usable = TIER_VRAM_GB.get(gpu_tier, 16.0) * USABLE_FRACTION
    total = weights + acts + logits + RESERVE_GB
    return {
        "weights_gb": round(weights, 2),
        "activations_gb": round(acts, 2),
        "logits_gb": round(logits, 2),
        "reserve_gb": RESERVE_GB,
        "total_gb": round(total, 2),
        "usable_gb": round(usable, 2),
        "fraction_of_usable": round(total / usable, 3),
        "fits": total <= usable,
        "source": "ESTIMATE",
    }

def seq_cap(gpu_tier, method, micro_batch=1):
    """Longest sequence whose ESTIMATE still fits the tier, also bounded by TIER_METHOD_CAP."""
    hard = TIER_METHOD_CAP.get(gpu_tier, TIER_METHOD_CAP["unknown"]).get(method, 4096)
    best = 1024
    for seq in range(1024, min(hard, SEQ_CEILING) + 1, 256):
        if estimate_vram_gb(gpu_tier, method, seq, micro_batch)["fits"]:
            best = seq
        else:
            break
    return min(best, hard)

def choose_method(gpu_tier, method, seq_wanted, micro_batch=1):
    """Tier-driven method decision; returns ``(method, notes)``.

    A T4 cannot hold 4B fp16 weights (8.8 GB) *plus* activations *plus* the logits term at 8192, so
    plain LoRA above the tier's LoRA cap becomes qlora instead of being launched into an OOM. Below
    that cap the user's explicit choice stands.
    """
    notes = []
    other = "qlora" if method == "lora" else "lora"
    cap_here = seq_cap(gpu_tier, method, micro_batch)
    if int(seq_wanted) <= cap_here:
        return method, notes
    cap_other = seq_cap(gpu_tier, other, micro_batch)
    here = estimate_vram_gb(gpu_tier, method, seq_wanted, micro_batch)
    if cap_other >= int(seq_wanted):
        there = estimate_vram_gb(gpu_tier, other, seq_wanted, micro_batch)
        notes.append(
            f"METHOD CHANGED {method} -> {other}: {method} at seq {seq_wanted} needs "
            f"{here['total_gb']} GB but {gpu_tier} has {here['usable_gb']} GB usable "
            f"(ESTIMATE: weights {here['weights_gb']} + activations {here['activations_gb']} + "
            f"logits {here['logits_gb']} + reserve {here['reserve_gb']} GB), so it would OOM before "
            f"the first optimizer step. {other} at the same seq needs {there['total_gb']} GB. "
            f"Set MAX_SEQ_LENGTH <= {cap_here} if you really want {method} on {gpu_tier}."
        )
        return other, notes
    notes.append(
        f"neither {method} nor {other} fits seq {seq_wanted} on {gpu_tier}; keeping {method} and "
        f"clamping to its cap {cap_here}"
    )
    return method, notes

def derive_run_config(gpu_tier, method, token_stats=None, max_seq_length_param=0, micro_batch_param=1,
                      accum_param=8, epochs_param=3, lr_param=2e-4):
    """Concrete, printable config. Every number is an ESTIMATE unless ``token_stats`` is measured.

    Sequence policy, so the user's explicit value is never downgraded silently: the requested
    ``MAX_SEQ_LENGTH`` is used as-is when it fits. When it cannot fit the tier the decision prints a
    ``!! MAX_SEQ_LENGTH`` line naming old -> new, the estimate behind it and how to keep the old
    value (smaller sequence, or a bigger tier). ``MAX_SEQ_LENGTH = 0`` means "take the recommendation".
    """
    notes = []
    requested_method = method
    micro = int(micro_batch_param)
    requested = int(max_seq_length_param)
    wanted = requested if requested > 0 else 8192
    method, method_notes = choose_method(gpu_tier, method, wanted, micro)
    notes.extend(method_notes)
    cap = seq_cap(gpu_tier, method, micro)

    if token_stats and token_stats.get("p99"):
        recommended = min(cap, next_multiple(token_stats["p99"] + 256))
        notes.append(f"seq recommended from measured p99={token_stats['p99']} -> {recommended} "
                     f"(tier cap for {method} = {cap})")
    else:
        recommended = min(cap, 8192)
        notes.append(f"seq from the tier only (no measurement yet) -> {recommended} "
                     f"(tier cap for {method} = {cap})")

    seq = requested if requested > 0 else recommended
    clamped = False
    if seq > cap:
        over = estimate_vram_gb(gpu_tier, method, seq, micro)
        notes.append(
            f"!! MAX_SEQ_LENGTH {seq} -> {cap}: {seq} does not fit {gpu_tier} with {method} "
            f"({over['total_gb']} GB vs {over['usable_gb']} GB usable, ESTIMATE). The requested value "
            f"is NOT used; {cap} is. Raise the tier, or lower MAX_SEQ_LENGTH yourself."
        )
        seq = cap
        clamped = True

    estimate = estimate_vram_gb(gpu_tier, method, seq, micro)
    if not estimate["fits"]:
        notes.append(f"{method} at seq {seq} still projects {estimate['total_gb']} GB: expect an OOM")
    elif estimate["fraction_of_usable"] > 0.85:
        notes.append(f"tight: {estimate['fraction_of_usable']:.0%} of usable VRAM ({method}, seq {seq}); "
                     f"section 7 measures the truth before any long run")
    if micro > 1 and seq > 4096:
        notes.append(f"micro_batch {micro} at seq {seq} is very likely too large for {gpu_tier}")
    accum = int(accum_param)
    if micro * accum != 8:
        notes.append(f"effective batch is {micro * accum} sequences, not the 8 the shipped configs assume")

    return {
        "gpu_tier": gpu_tier,
        "method_requested": requested_method,
        "method": method,                 # lora | qlora; both train LoRA adapters
        "quantized_base": method == "qlora",
        "max_seq_length_requested": requested,
        "max_seq_length": seq,
        "seq_clamped": clamped,
        "seq_cap": cap,
        "micro_batch": micro,
        "grad_accum": accum,
        "effective_batch": micro * accum,
        "epochs": float(epochs_param),
        "learning_rate": float(lr_param),
        "precision": PRECISION,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "gradient_checkpointing": True,
        "vram_estimate": estimate,
        "notes": notes,
        "source": "measured" if token_stats and token_stats.get("method") == "tokenizer" else "estimate",
    }

def print_derived(d):
    label = "measured token lengths" if d["source"] == "measured" else "ESTIMATE, no local dataset yet"
    print(f"Derived effective configuration ({label})")
    for key in ("gpu_tier", "method", "quantized_base", "max_seq_length", "micro_batch", "grad_accum",
                "effective_batch", "epochs", "learning_rate", "precision", "lora_r", "lora_alpha",
                "lora_dropout", "gradient_checkpointing"):
        print(f"  {key:<22}= {d[key]}")
    est = d["vram_estimate"]
    print(f"  {'vram ESTIMATE':<22}= {est['total_gb']} GB of {est['usable_gb']} GB usable "
          f"[weights {est['weights_gb']} + activations {est['activations_gb']} + "
          f"logits {est['logits_gb']} + reserve {est['reserve_gb']} GB] -> "
          f"{'fits' if est['fits'] else 'DOES NOT FIT'} ({est['fraction_of_usable']:.0%})")
    if d["method"] != d["method_requested"]:
        print(f"  !! method          {d['method_requested']} -> {d['method']} (reason in the notes)")
    if d["seq_clamped"]:
        print(f"  !! max_seq_length  {d['max_seq_length_requested']} -> {d['max_seq_length']} "
              f"(reason in the notes)")
    for note in d["notes"]:
        print(f"  note: {note}")

RUN = derive_run_config(GPU_TIER, TRAINING_METHOD, None, MAX_SEQ_LENGTH, MICRO_BATCH_SIZE,
                        GRADIENT_ACCUMULATION_STEPS, NUM_EPOCHS, LEARNING_RATE)
print_derived(RUN)
if RUN["method"] != TRAINING_METHOD:
    # Sections 3-8 all read these globals, so the run must carry the decision the tier made.
    print(f"applying the tier decision: TRAINING_METHOD {TRAINING_METHOD} -> {RUN['method']}")
    TRAINING_METHOD = RUN["method"]

## 3. Load the repo

Purpose: get the code that owns the training loop, checkpoint format and evaluation, pin it, and
prove it imports. The notebook never re-implements the loop. Cost: ~1-2 min, network only.

`pip install -e .` alone installs **no** dependencies — the package is stdlib-only by design — so
the ML stack is checked/installed separately, and only what is missing.


In [ ]:
# --- 3a. Clone / update / pin the repo --------------------------------------
REPO_URL = "https://github.com/Le-Minh-Nhut/zaloai"
REPO_DIR = Path("/content/zaloai") if IN_COLAB else Path.cwd()


def _git(*args, cwd):
    out = subprocess.run(["git", *args], cwd=str(cwd), capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed in {cwd}:\n{out.stderr.strip()}")
    return out.stdout.strip()


if not (REPO_DIR / "scripts" / "train.py").is_file():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but does not contain scripts/train.py. Point REPO_DIR at the clone "
            "or delete that directory and re-run this cell."
        )
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"repo already present: {REPO_DIR}")

if BRANCH_OR_COMMIT:
    _git("fetch", "--all", "--tags", "--prune", cwd=REPO_DIR)
    try:
        _git("checkout", BRANCH_OR_COMMIT, cwd=REPO_DIR)
    except RuntimeError as exc:
        raise RuntimeError(
            f"cannot check out {BRANCH_OR_COMMIT!r}. Use a branch name (main) or a full commit sha "
            f"that exists on {REPO_URL}. Original error:\n{exc}"
        ) from None
print("HEAD:", _git("rev-parse", "--short", "HEAD", cwd=REPO_DIR), "| ref:", BRANCH_OR_COMMIT or "(unchanged)")


In [ ]:

# --- 3b. Install the repo itself (editable) --------------------------------
# Dependencies were handled in section 1c. The package declares none (stdlib-only by design), so
# `--no-deps` is correct here: it installs the console script + the src/ layout without dragging in
# a second copy of torch.
def _pip(*args):
    print("pip", " ".join(args), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

_pip("-e", str(REPO_DIR), "--no-deps")
print("installed lifemate from", REPO_DIR)

# QLoRA needs bitsandbytes; it is installed here (not in section 1) because only section 2 knows the
# chosen method. A missing install stays a hard error in section 6 — never a silent fp16 fallback.
if TRAINING_METHOD == "qlora" and not has_module("bitsandbytes"):
    _pip("bitsandbytes>=0.43"); print("installed bitsandbytes for QLoRA")


In [ ]:

# --- 3c. Verify imports + bind the repo CLI contract -----------------------
import threading  # noqa: E402 - used by run_cli below

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import lifemate  # noqa: E402
from lifemate.config import TUNING_MODES, dump_config, load_config  # noqa: E402
from lifemate.errors import LifeMateError  # noqa: E402

if Path.cwd() != REPO_DIR:
    os.chdir(REPO_DIR)  # scripts/ resolve configs/ and outputs/ relative to cwd

print(f"lifemate {lifemate.__version__} from {Path(lifemate.__file__).parent}")
print(f"TUNING_MODES in this checkout: {TUNING_MODES}")
import peft  # noqa: E402
import transformers  # noqa: E402
print(f"transformers {transformers.__version__} | peft {peft.__version__}")


def script_help(script_name):
    """Raw ``--help`` text of a repo script; fails loudly when it is absent or broken."""
    script = REPO_DIR / "scripts" / script_name
    if not script.is_file():
        raise RuntimeError(f"{script} is missing; does BRANCH_OR_COMMIT point at a commit that has it?")
    out = subprocess.run([sys.executable, str(script), "--help"], cwd=str(REPO_DIR),
                         capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(f"`python {script.name} --help` failed:\n{out.stdout}\n{out.stderr}")
    return out.stdout + out.stderr


def script_flags(script_name):
    """Flags a script actually accepts, parsed from its own help text (never guessed)."""
    return set(re.findall(r"(?<![\w-])(--[a-z0-9][a-z0-9-]*)", script_help(script_name)))


def require_flags(script_name, flags, hint=""):
    """Fail with an actionable message when the checkout lacks a flag this notebook uses."""
    have = script_flags(script_name)
    missing = [flag for flag in flags if flag not in have]
    if missing:
        raise RuntimeError(
            f"{script_name} does not accept {missing}. It accepts: {', '.join(sorted(have))}. "
            f"{hint} Pin BRANCH_OR_COMMIT to a commit that has them, or update the repo."
        )
    return have


def flag_choices(script_name, flag):
    """Allowed values of a choice flag, from the script's own help text."""
    match = re.search(rf"{re.escape(flag)}\s*\{{([^}}]*)\}}", script_help(script_name))
    return [part.strip() for part in match.group(1).split(",")] if match else []


TUNING_CHOICES = flag_choices("train.py", "--tuning")
QLORA_SUPPORTED = "qlora" in TUNING_CHOICES
print(f"--tuning accepts: {TUNING_CHOICES} (qlora supported: {QLORA_SUPPORTED})")
# The mode the repo will actually run with. Under QLoRA the trained artifact is still a LoRA
# adapter — only the frozen base is 4-bit — so adapter semantics are identical.
CLI_TUNING_MODE = "qlora" if (TRAINING_METHOD == "qlora" and QLORA_SUPPORTED) else "lora"
if TRAINING_METHOD == "qlora" and not QLORA_SUPPORTED:
    print(
        "warning: TRAINING_METHOD=qlora requested, but this checkout has no `--tuning qlora`. "
        "Section 6 still verifies the 4-bit stack (NF4 + prepare_model_for_kbit_training, adapter "
        "gradients flowing), then stops with an actionable error instead of launching a run this "
        "trainer cannot honour."
    )

LAST_CLI_PEAK_MIB = None


def gpu_used_mib():
    """Device memory in use, from nvidia-smi (sees other processes), else torch counters."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        if out.returncode == 0 and out.stdout.strip():
            return str(int(out.stdout.split()[0]))
    except Exception:
        pass
    if torch is not None and torch.cuda.is_available():
        return str(torch.cuda.memory_allocated() // 2**20)
    return "n/a"


def stream_process(cmd, env, *, sample_peak, peak_interval, status_prefix, on_tick=None):
    """Run ``cmd``, stream stdout, sample device memory; return (rc, lines).

    ``on_tick`` runs every ``peak_interval`` seconds while the child is alive — that is the hook the
    training cell uses to mirror finished checkpoints to Drive *during* the run.
    """
    global LAST_CLI_PEAK_MIB
    print("$", " ".join(str(part) for part in cmd), flush=True)
    LAST_CLI_PEAK_MIB = None
    started = time.time()
    lines = []
    proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)

    def _pump():
        for line in proc.stdout:
            text = line.rstrip("\n")
            lines.append(text)
            print(text, flush=True)

    reader = threading.Thread(target=_pump, daemon=True)
    reader.start()
    last_status = 0.0
    while proc.poll() is None:
        if sample_peak:
            used = gpu_used_mib()
            if used.isdigit():
                LAST_CLI_PEAK_MIB = max(LAST_CLI_PEAK_MIB or 0, int(used))
        if on_tick is not None:
            on_tick()
        if time.time() - last_status >= max(peak_interval, 30):
            last_status = time.time()
            print(f"    [{status_prefix}] {int(time.time() - started)}s elapsed, "
                  f"{gpu_used_mib()} MiB used", flush=True)
        time.sleep(peak_interval)
    reader.join(timeout=30)
    rc = proc.wait()
    if sample_peak and LAST_CLI_PEAK_MIB:
        print(f"    [{status_prefix}] sampled peak: {LAST_CLI_PEAK_MIB} MiB", flush=True)
    return rc, lines


def run_cli(script_name, args, env_extra=None, sample_peak=False, peak_interval=2.0):
    """Run a repo script; raise with the script's own exit-code meaning on a non-zero exit."""
    env = dict(os.environ)
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("PYTHONUNBUFFERED", "1")
    if env_extra:
        env.update({k: str(v) for k, v in env_extra.items()})
    cmd = [sys.executable, str(REPO_DIR / "scripts" / script_name), *[str(a) for a in args]]
    rc, lines = stream_process(cmd, env, sample_peak=sample_peak, peak_interval=peak_interval,
                               status_prefix="cli")
    if rc != 0:
        raise RuntimeError(
            f"{script_name} exited {rc} (0 ok, 1 failure, 2 data missing, 3 environment or weights "
            "unusable, 130 interrupted). The actionable line is in the output above — the scripts "
            "print it immediately before exiting."
        )
    return lines


def free_gpu():
    """Collect garbage, empty the CUDA cache and reset peak counters."""
    import gc

    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    print(f"gpu after free: {gpu_used_mib()} MiB used")


print("helpers ready")


## 4. Google Drive: paths, dataset, atomic artifact sync

Purpose: decide where durable things live. Convention used below:

* `/content/...` (local, fast, ephemeral) — HF cache, training output, probe checkpoints;
* `MyDrive/lifemate_colab/` (durable) — `data/`, `datasets/`, `checkpoints/`, `outputs/`, `logs/`.

A Colab disconnect erases the runtime, not Drive. So checkpoints are written locally by the repo
(atomic staging + rename) and then copied to Drive through a staging directory, `fsync`ed, and
renamed — plus a `SYNC_COMPLETE.json` marker. Resume refuses any Drive checkpoint without that
marker, because a torn copy looks exactly like a valid directory.

Cost: ~1 CU-minute, plus transfer time proportional to checkpoint size (~35 MB per LoRA step dir).

No personal path is hardcoded: `DRIVE_ROOT` is resolved from the mounted Drive root at runtime.


In [ ]:
# --- 4a. Mount Drive (optional) and define every path in one place ----------
DRIVE_ROOT = None
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        print("note: USE_GOOGLE_DRIVE is on but this is not Colab; falling back to local paths")
    else:
        from google.colab import drive  # noqa: PLC0415 - Colab only
        drive.mount("/content/drive")
        DRIVE_ROOT = Path("/content/drive/MyDrive/lifemate_colab")

LOCAL_WORK = Path("/content/lifemate_work") if IN_COLAB else Path(REPO_DIR) / "outputs" / "colab_work"
DATA_ROOT = LOCAL_WORK / "data"   # outside the git clone: the repo tree stays clean
LOCAL_DATASET_ROOT = LOCAL_WORK / "datasets"
LOCAL_CKPT_ROOT = LOCAL_WORK / "checkpoints"
LOCAL_OUTPUT_ROOT = LOCAL_WORK / "outputs"
HF_CACHE = LOCAL_WORK / "hf_cache"

for _p in (LOCAL_WORK, LOCAL_DATASET_ROOT, LOCAL_CKPT_ROOT, LOCAL_OUTPUT_ROOT, HF_CACHE):
    _p.mkdir(parents=True, exist_ok=True)

# One HF cache for the whole session: weights are downloaded once, then reused.
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

DRIVE_PATHS = {}
if DRIVE_ROOT is not None:
    DRIVE_PATHS = {
        "data": DRIVE_ROOT / "data",
        "datasets": DRIVE_ROOT / "datasets",
        "checkpoints": DRIVE_ROOT / "checkpoints",
        "outputs": DRIVE_ROOT / "outputs",
        "logs": DRIVE_ROOT / "logs",
    }
    for _p in DRIVE_PATHS.values():
        _p.mkdir(parents=True, exist_ok=True)
    print("drive root:", DRIVE_ROOT)

print(f"local work : {LOCAL_WORK}")
print(f"repo       : {REPO_DIR}")
print(f"dataset in : {DATA_ROOT} (provisioned in 4c; test splits and test labels are never read)")


In [ ]:
# --- 4b. Atomic artifact sync + completion markers --------------------------
SYNC_MARKER = "SYNC_COMPLETE.json"
_PROHIBITED = re.compile(r"(^|/)[^/]+_environment(/|$)")


def _reject_prohibited(path):
    """Refuse any path under an organizers' ``*_environment`` directory."""
    if _PROHIBITED.search(str(Path(path)).replace("\\", "/")):
        raise PermissionError(
            f"{path} is inside a *_environment directory: off-limits for reading, copying or "
            "uploading (docs/COMPETITION_COMPLIANCE.md section 1)."
        )
    return Path(path)


def safe_unzip(archive, dest):
    """Extract an archive, skipping any member under an organizers' ``*_environment`` dir."""
    import zipfile

    archive, dest = _reject_prohibited(archive), Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    skipped = 0
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            name = member.filename.replace("\\", "/")
            if _PROHIBITED.search(name) or name.startswith("/") or ".." in Path(name).parts:
                skipped += 1
                continue
            target = dest / name
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(member) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
    print(f"extracted {archive} -> {dest} (skipped {skipped} prohibited/unsafe member(s))")
    return dest


def _fsync_tree(root):
    for item in Path(root).rglob("*"):
        if item.is_file():
            with open(item, "rb") as handle:
                os.fsync(handle.fileno())


def file_manifest(root):
    root = Path(root)
    return {
        str(p.relative_to(root)): p.stat().st_size
        for p in sorted(root.rglob("*")) if p.is_file()
    }


def write_sync_marker(root, extra=None):
    """Completion marker: written last, atomically, inside the copied tree."""
    root = _reject_prohibited(root)
    payload = {
        "written_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "host": "colab" if IN_COLAB else "local",
        "files": file_manifest(root),
    }
    payload.update(extra or {})
    tmp = root / (SYNC_MARKER + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    with open(tmp, "rb") as handle:
        os.fsync(handle.fileno())
    os.replace(tmp, root / SYNC_MARKER)
    return root / SYNC_MARKER


def atomic_sync(src, dst, extra=None):
    """Copy ``src`` to ``dst`` via a staging dir + fsync + rename (+ marker).

    An interrupted copy leaves ``<dst>.incomplete-*`` behind and never a valid-looking ``dst``.
    """
    src, dst = _reject_prohibited(src), _reject_prohibited(dst)
    if not src.is_dir():
        raise FileNotFoundError(f"nothing to sync: {src} is not a directory")
    staging = dst.with_name(f"{dst.name}.incomplete-{int(time.time())}")
    if staging.exists():
        shutil.rmtree(staging)
    shutil.copytree(src, staging)
    _fsync_tree(staging)
    write_sync_marker(staging, extra={**(extra or {}), "source": str(src)})
    if dst.exists():
        shutil.rmtree(dst)  # same-filesystem replace of a non-empty dir is not atomic on POSIX
    os.replace(staging, dst)
    print(f"synced {src} -> {dst} ({len(file_manifest(dst))} files)")
    return dst


def checkpoint_ready(path, require_marker=False):
    """Can the repo's trainer genuinely resume from this directory?

    Returns ``(ok, reason)``. ``require_marker`` must be True for anything that crossed a network
    copy (Drive), because only the marker proves the copy finished.
    """
    path = Path(path)
    if not path.is_dir():
        return False, f"{path} is not a directory"
    if require_marker and not (path / SYNC_MARKER).is_file():
        return False, f"{path} has no {SYNC_MARKER}; the copy never completed"
    state = path / "training_state.json"
    if not state.is_file():
        return False, f"{path} has no training_state.json (not a step-<n> checkpoint)"
    try:
        payload = json.loads(state.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        return False, f"training_state.json is corrupt: {exc}"
    if int(payload.get("step") or 0) <= 0:
        return False, f"training_state.json records step {payload.get('step')!r}"
    weights = list(path.glob("adapter_model.*")) + list(path.glob("*.safetensors")) + list(path.glob("pytorch_model*.bin"))
    if not weights:
        return False, f"{path} has no weights (adapter_model.* / *.safetensors / pytorch_model*.bin)"
    if not (path / "optimizer_state.pt").is_file():
        return False, (
            f"{path} has no optimizer_state.pt: resuming would restart AdamW moments from zero. "
            "That is adapter-only loading, not optimizer-state resumption."
        )
    return True, "ok"


In [ ]:

# --- 4c. Provide the dataset: reuse, copy from Drive, unzip, or upload -----
DRIVE_DATA_ZIP = ""  # optional: path to the organizers' archive already on Drive
DATASET_UPLOAD_NAME = "dataset.zip"   # filename the upload widget will ask for
SPLIT_FILES = ("train.jsonl", "train_tools_catalog.json")


def _visible_subdirs(path):
    """Child directories of ``path``, skipping any ``*_environment`` name (never entered)."""
    try:
        names = sorted(os.listdir(path))
    except OSError:
        return []
    children = []
    for name in names:
        if _PROHIBITED.search(f"{path}/{name}/"):
            continue
        child = Path(path) / name
        if child.is_dir():
            children.append(child)
    return children


def locate_data_root(base, max_depth=2):
    """Directory holding ``train/train.jsonl``, at most ``max_depth`` levels below ``base``.

    Breadth-first and bounded: it never recurses, and a ``*_environment`` directory is filtered out
    before ``is_dir()`` is even called on it.
    """
    level = [_reject_prohibited(base)]
    for _depth in range(max_depth + 1):
        for candidate in level:
            if (candidate / "train" / "train.jsonl").is_file():
                return candidate
        level = [child for parent in level for child in _visible_subdirs(parent)]
    return None


def copy_train_split(src_root, dst_root):
    """Copy only the named files/dirs of the ``train`` split — never list a *_environment dir."""
    src_split, dst_split = Path(src_root) / "train", Path(dst_root) / "train"
    dst_split.mkdir(parents=True, exist_ok=True)
    for name in SPLIT_FILES:
        if (src_split / name).is_file():
            shutil.copy2(src_split / name, dst_split / name)
    for name in ("audio", "zaic_tools"):
        src, dst = src_split / name, dst_split / name
        if src.is_dir() and not dst.exists():
            shutil.copytree(src, dst)
    return dst_split


def provision_dataset():
    """Ensure ``DATA_ROOT/train/train.jsonl`` exists; return its resolved root."""
    found = locate_data_root(DATA_ROOT)
    if found is not None:
        return found
    if DRIVE_ROOT is not None and locate_data_root(DRIVE_PATHS["data"]):
        copy_train_split(locate_data_root(DRIVE_PATHS["data"]), DATA_ROOT)
        print(f"copied the train split from Drive ({DRIVE_PATHS['data']})")
    elif str(DRIVE_DATA_ZIP).strip():
        safe_unzip(Path(str(DRIVE_DATA_ZIP).strip()).expanduser(), DATA_ROOT)
    elif IN_COLAB:
        from google.colab import files  # noqa: PLC0415 - Colab only
        print(f"Upload the LifeMate archive as {DATASET_UPLOAD_NAME}. Only `train/` is used; "
              "the test splits are never read.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("nothing uploaded; re-run this cell and pick the dataset archive")
        name = next(iter(uploaded))
        archive = Path("/content") / name
        if archive.suffix == ".zip":
            safe_unzip(archive, DATA_ROOT)
        else:
            raise RuntimeError(
                f"uploaded {name} is not a .zip archive. Upload the LifeMate data archive (or set "
                "DRIVE_DATA_ZIP to its path on Drive) so the notebook can place train/ under "
                f"{DATA_ROOT}."
            )
    else:
        raise RuntimeError(
            f"{DATA_ROOT}/train/train.jsonl not found and no source configured. Provide the official "
            f"LifeMate data tree (train/train.jsonl, train/train_tools_catalog.json, train/audio/) "
            f"under {DATA_ROOT}, or set DRIVE_DATA_ZIP."
        )
    root = locate_data_root(DATA_ROOT)
    if root is None:
        raise RuntimeError(
            f"the archive was unpacked but no train/train.jsonl was found under {DATA_ROOT}. It "
            "probably nests the split differently; set DATA_ROOT to the directory that contains "
            "`train/`, then re-run this cell."
        )
    return root


DATA_ROOT = provision_dataset()
print("data root      :", DATA_ROOT)
print("train split    :", DATA_ROOT / "train" / "train.jsonl")
print("tool catalog   :", (DATA_ROOT / "train" / "train_tools_catalog.json").is_file())


## 5. Dataset preparation + measured token lengths

Purpose: reuse `scripts/prepare_dataset.py` (it owns the trajectory/prompt build, the target-value
visibility check and the truncation accounting), then **measure** the token-length distribution on a
CPU sample so the sequence budget is a measurement, not a guess. Cost: 1-3 min CPU, no GPU, no CU
beyond the idle runtime.

What is produced, and what is deliberately never produced:

* `train.jsonl` / `val.jsonl` decision points split by the repo's own group-aware splitter;
* `manifest.json` / `stats.json` with the truncation and drop reasons;
* no test labels anywhere: only `<data-root>/train/` is read, `public_test` is never touched.

Rebuilding is cheap and deterministic (same seed → same split), so this cell re-runs the fit check
once the measured p99 has shifted the budget.


In [ ]:
# --- 5a. Build (or reuse) the SFT dataset and measure token lengths ---------
import random  # deterministic sampling of records for the token measurement

TRAIN_DATASET_DIR = LOCAL_DATASET_ROOT / "train"
MANIFEST_PATH = TRAIN_DATASET_DIR / "manifest.json"
TOKENIZER_CACHE = {}


def prepare_dataset(seq_budget):
    """Run the repo's data builder at one sequence budget; returns the manifest dict."""
    args = [
        "--data-root", DATA_ROOT,
        "--split", "train",
        "--output", LOCAL_DATASET_ROOT,
        "--tokenizer", MODEL_ID,
        "--max-seq-length", int(seq_budget),
        "--val-ratio", 0.1,
        "--seed", SEED,
    ]
    require_flags("prepare_dataset.py", ["--data-root", "--split", "--output", "--tokenizer",
                                         "--max-seq-length", "--val-ratio", "--seed"])
    run_cli("prepare_dataset.py", args)
    if not MANIFEST_PATH.is_file():
        raise RuntimeError(
            f"{MANIFEST_PATH} was not written; inspect the builder output above. It refuses to "
            "silently continue when the split has no tool catalog."
        )
    return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))


def measured_token_stats(max_records=300):
    """Tokenise a fixed sample of records and report the real length distribution.

    Uses the repo's own ``build_supervised_example`` at a 32k budget so the numbers are
    *untruncated* lengths in the exact tokenisation training will use. CPU only.
    """
    from lifemate.training.collator import build_supervised_example
    from transformers import AutoTokenizer

    path = TRAIN_DATASET_DIR / "train.jsonl"
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not records:
        raise RuntimeError(f"{path} holds no SFT records")
    if MODEL_ID not in TOKENIZER_CACHE:
        TOKENIZER_CACHE[MODEL_ID] = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer = TOKENIZER_CACHE[MODEL_ID]

    rng = random.Random(SEED)
    sample = records if len(records) <= max_records else rng.sample(records, max_records)
    lengths, dropped = [], {}
    for record in sample:
        example = build_supervised_example(
            tokenizer, list(record.get("messages") or []), str(record.get("target") or ""), 32768
        )
        if not example.get("kept"):
            dropped[example.get("reason") or "unknown"] = dropped.get(example.get("reason") or "unknown", 0) + 1
            continue
        lengths.append(example["n_prompt_tokens"] + example["n_target_tokens"])
    if not lengths:
        raise RuntimeError(f"no tokenisable record in the {len(sample)}-record sample: {dropped}")
    lengths.sort()

    def pct(q):
        return int(lengths[min(len(lengths) - 1, int(q * (len(lengths) - 1)))])

    return {
        "method": "tokenizer",
        "tokenizer": MODEL_ID,
        "n_sampled": len(sample),
        "n_tokenised": len(lengths),
        "n_unusable": len(sample) - len(lengths),
        "unusable_reasons": dropped,
        "p50": pct(0.50),
        "p90": pct(0.90),
        "p99": pct(0.99),
        "max": lengths[-1],
        "mean": int(sum(lengths) / len(lengths)),
    }


def existing_manifest():
    """The manifest already on disk, when it was built by this model at this budget."""
    if not MANIFEST_PATH.is_file():
        return None
    try:
        payload = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return None
    if not (TRAIN_DATASET_DIR / "train.jsonl").is_file():
        return None
    return payload


def manifest_is_reusable(payload, budget):
    """True only when the existing manifest describes *this* tokenizer at *this* budget.

    The truncation counts are a function of both, so a manifest from another budget would make
    the pre-training report describe a different run.
    """
    if not payload or DATASET_MODE == "rebuild":
        return False
    fit = payload.get("fit") or {}
    same_budget = fit.get("max_seq_length") == int(budget)
    same_tokenizer = MODEL_ID.split("/")[-1] in str(payload.get("tokenization") or "")
    same_data = str(payload.get("data_root") or "").rstrip("/") == str(Path(DATA_ROOT).resolve()).rstrip("/")
    return bool(same_budget and same_tokenizer and same_data and fit.get("method") == "tokenizer")


# The tier-derived budget from 2b: it already honours the requested MAX_SEQ_LENGTH or the clamp
# that cell printed. The measured p99 below may still shrink it, which rebuilds the fit numbers.
FIRST_BUDGET = int(RUN["max_seq_length"])
manifest = existing_manifest()
if manifest_is_reusable(manifest, FIRST_BUDGET):
    print(f"reusing the prepared dataset at seq {FIRST_BUDGET} (DATASET_MODE={DATASET_MODE})")
else:
    manifest = prepare_dataset(FIRST_BUDGET)

TOKEN_STATS = measured_token_stats()
RUN = derive_run_config(GPU_TIER, TRAINING_METHOD, TOKEN_STATS, MAX_SEQ_LENGTH, MICRO_BATCH_SIZE,
                        GRADIENT_ACCUMULATION_STEPS, NUM_EPOCHS, LEARNING_RATE)
if int(RUN["max_seq_length"]) != int(manifest["fit"]["max_seq_length"]):
    # The measured p99 moved the budget: the truncated/dropped counts must be recomputed at the
    # budget we will actually train with, or the report describes a different run.
    print(f"measured p99 moved the budget to {RUN['max_seq_length']}: rebuilding the fit numbers")
    manifest = prepare_dataset(RUN["max_seq_length"])
print_derived(RUN)


In [ ]:
# --- 5b. Dataset report printed before any training ------------------------
fit = manifest.get("fit") or {}
fit_val = manifest.get("fit_val") or {}
n_train = len((TRAIN_DATASET_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines())
n_val = len((TRAIN_DATASET_DIR / "val.jsonl").read_text(encoding="utf-8").splitlines())
if fit.get("method") != "tokenizer":
    raise RuntimeError(
        "the manifest's fit block came from the chars/4 ESTIMATE, not a tokenizer. Re-run section 5a "
        "with a reachable tokenizer before trusting the truncation numbers."
    )

DATASET_REPORT = {
    "dataset_dir": str(TRAIN_DATASET_DIR),
    "n_train_records": n_train,
    "n_val_records": n_val,
    "effective_max_seq_length": int(manifest["fit"]["max_seq_length"]),
    "train": {
        "n_truncated": fit.get("n_truncated"),
        "n_truncated_fraction": fit.get("n_truncated_fraction"),
        "n_dropped": fit.get("n_dropped"),
        "dropped_reasons": fit.get("dropped_reasons"),
    },
    "val": {
        "n_truncated": fit_val.get("n_truncated"),
        "n_dropped": fit_val.get("n_dropped"),
        "dropped_reasons": fit_val.get("dropped_reasons"),
    },
    "token_lengths_sampled": {k: TOKEN_STATS[k] for k in ("n_sampled", "n_tokenised", "p50", "p90", "p99", "max", "mean")},
    "unusable_in_sample": TOKEN_STATS["unusable_reasons"],
    "split_manifest": {
        "group_hash": manifest.get("group_hash"),
        "n_train_samples": manifest.get("n_train"),
        "n_val_samples": manifest.get("n_val"),
        "seed": manifest.get("seed"),
        "ratio": manifest.get("ratio"),
    },
}

if n_train == 0:
    raise RuntimeError(f"{TRAIN_DATASET_DIR}/train.jsonl is empty")
if n_train + n_val == 0:
    raise RuntimeError(
        f"no usable record at seq {DATASET_REPORT['effective_max_seq_length']}. Read the dropped "
        f"reasons above ({DATASET_REPORT['train']['dropped_reasons']}); raising the sequence budget is "
        "the fix when targets overflow it, not lowering it."
    )
if n_train == 0 or n_val == 0:
    raise RuntimeError(
        f"split produced {n_train} train / {n_val} val records; both must be non-empty for a "
        "validation-loss signal and a meaningful resume check."
    )

print(json.dumps(DATASET_REPORT, indent=2, ensure_ascii=False))
print(
    "\nreading of the numbers: 'truncated' records are still trained on (middle-truncated prompt, "
    "request and tool catalog preserved); 'dropped' records were refused because the target could "
    "not be supervised — a dropped target is never replaced by a synthesised label."
)


## 6. Load Qwen3.5-4B, verify compatibility, attach the adapter

Purpose: prove the checkpoint is real, that this `transformers` release can build it, that the LoRA
targets are discovered **from the architecture** (Qwen's hybrid linear-attention projections
included — never a hardcoded Llama q/k/v/o/gate/up/down list), and that gradients actually reach the
adapter. Cost: 3-10 min on GPU (weights ~8.7 GB from the HF hub, cached afterwards) — the single
largest fixed cost after training itself.

QLoRA specifics: the base goes into 4-bit NF4 with the **compute** dtype matching the platform
(bf16 everywhere except T4, which uses fp16), `prepare_model_for_kbit_training` is applied so the
frozen 4-bit base still propagates adapter gradients, and a missing `bitsandbytes` is a hard error.

The model is loaded here for verification and then released: the trainer runs as a subprocess in
section 8, so holding a second copy would just halve the VRAM available to it.

On a GPU without bf16 (T4) this section prints the fp16 warning again, next to the config it is
about: fp16 relies on `torch.amp.GradScaler` for loss scaling in the repo trainer, a T4 user should
prefer `TRAINING_METHOD="qlora"`, and `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` (exported in
section 1) is what keeps allocator fragmentation down at long sequence lengths.


In [ ]:

# --- 6a. Effective config: one YAML the repo itself parses ------------------
CONFIG_DIR = LOCAL_OUTPUT_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / (f"colab_{CLI_TUNING_MODE}" + f"_seq{RUN['max_seq_length']}.yaml")
CKPT_OUTPUT_DIR = LOCAL_CKPT_ROOT
RUN_DIR = CKPT_OUTPUT_DIR / RUN_NAME

def config_overrides():
    """Repo config overrides for this run.

    ``training.quantization`` must agree with ``training.tuning_mode`` — the dataclass
    rejects a qlora run whose quantization is "none" — and both are part of the config
    fingerprint, so a resume across a mismatch is refused by the trainer itself.
    """
    return {
        "data.root": str(DATA_ROOT),
        "model.name": MODEL_ID,                     # explicit: the repo default is a different model
        "model.device": "cuda",
        "model.torch_dtype": PRECISION,
        "training.tuning_mode": CLI_TUNING_MODE,    # runtime-owned, never an opaque value
        "training.quantization": "nf4" if CLI_TUNING_MODE == "qlora" else "none",
        "training.precision": PRECISION,
        "training.output_dir": str(CKPT_OUTPUT_DIR),
        "training.max_seq_length": int(RUN["max_seq_length"]),
        "training.per_device_train_batch_size": int(RUN["micro_batch"]),
        "training.gradient_accumulation_steps": int(RUN["grad_accum"]),
        "training.num_train_epochs": float(RUN["epochs"]),
        "training.learning_rate": float(RUN["learning_rate"]),
        "training.weight_decay": 0.01,
        "training.warmup_ratio": 0.03,
        "training.lr_scheduler_type": "cosine",
        "training.gradient_checkpointing": True,
        "training.gradient_clip_norm": 1.0,
        "training.logging_steps": 10,
        "training.save_strategy": "epoch",
        "training.eval_strategy": "epoch",
        "training.save_total_limit": 2,
        "training.seed": int(SEED),
        "training.dataloader_num_workers": 0,
        "training.report_to": [],                   # W&B is opt-in, never required
        "lora.r": int(RUN["lora_r"]),
        "lora.alpha": int(RUN["lora_alpha"]),
        "lora.dropout": float(RUN["lora_dropout"]),
        "lora.target_modules": None,                # None => discovery from the real module names
        "lora.bias": "none",
    }


def build_config():
    """Load the repo base config + overrides, prove the invariants, dump for the CLIs."""
    cfg = load_config(str(REPO_DIR / "configs" / "base.yaml"), config_overrides())
    if cfg.model.name != MODEL_ID:
        raise RuntimeError(f"config model.name is {cfg.model.name!r}, expected {MODEL_ID!r}")
    if cfg.training.tuning_mode != CLI_TUNING_MODE:
        raise RuntimeError(
            f"config tuning_mode is {cfg.training.tuning_mode!r}, expected {CLI_TUNING_MODE!r}"
        )
    dump_config(cfg, CONFIG_PATH)
    return cfg


import math  # noqa: E402 - total-step estimate below

from lifemate.training.trainer import config_fingerprint, optimizer_steps_per_epoch  # noqa: E402

cfg = build_config()
CONFIG_HASH = config_fingerprint(cfg)
print(f"config written : {CONFIG_PATH}")
print(f"config hash    : {CONFIG_HASH}  (a resume must match this)")
print(f"tuning_mode    : {cfg.training.tuning_mode} | quantization {cfg.training.quantization} "
      f"| precision {cfg.training.precision} | seq {cfg.training.max_seq_length}")
print(f"batch          : micro {cfg.training.per_device_train_batch_size} x accum "
      f"{cfg.training.gradient_accumulation_steps} = "
      f"{cfg.training.per_device_train_batch_size * cfg.training.gradient_accumulation_steps} sequences")
if PRECISION == "float16":
    print(
        "\n!! PRECISION=float16 (no bf16 on this GPU): fp16 training relies on torch.amp.GradScaler\n"
        "   inside the repo trainer for loss scaling, and that fp16+scaler path is NOT verified here.\n"
        f"   This run is tuning_mode={CLI_TUNING_MODE} at seq {cfg.training.max_seq_length}. On a T4\n"
        "   prefer TRAINING_METHOD=\"qlora\" (cell 2b derives it when 16-bit LoRA cannot fit); the\n"
        "   4-bit base is what makes 8192 tokens fit in 16 GB.\n"
    )

STEPS_PER_EPOCH = optimizer_steps_per_epoch(
    DATASET_REPORT["n_train_records"],
    cfg.training.per_device_train_batch_size,
    cfg.training.gradient_accumulation_steps,
)
TOTAL_STEPS = int(math.ceil(STEPS_PER_EPOCH * cfg.training.num_train_epochs))
print(f"steps/epoch    : {STEPS_PER_EPOCH} -> horizon {TOTAL_STEPS} "
      f"(MAX_STEPS={MAX_STEPS or 'unset'} caps it)")


In [ ]:

# --- 6b. Load the checkpoint through the repo loader, verify adapter + grads --
import inspect  # noqa: E402

from lifemate.models.loader import count_parameters, load_model_and_tokenizer  # noqa: E402
from lifemate.training.lora_finetuning import discover_target_modules  # noqa: E402

COMPUTE_DTYPE = torch.bfloat16 if PRECISION == "bfloat16" else torch.float16

LOADER_KWARGS = set(inspect.signature(load_model_and_tokenizer).parameters)
if "quantization" not in LOADER_KWARGS:
    raise RuntimeError(
        "this checkout's load_model_and_tokenizer has no `quantization` argument, so the repo's "
        f"4-bit path is absent (it accepts: {sorted(LOADER_KWARGS)}). Pin BRANCH_OR_COMMIT to a "
        "commit that has the QLoRA path, or set TRAINING_METHOD=lora (16-bit adapters). Section 6 "
        "will not hand-roll a BitsAndBytesConfig that the trainer's own loader would never see."
    )
if CLI_TUNING_MODE == "qlora":
    from lifemate.models.loader import require_bitsandbytes  # noqa: PLC0415 - QLoRA only

    bnb = require_bitsandbytes()   # raises LifeMateError with the pip command when absent
    print(f"bitsandbytes   : {getattr(bnb, '__version__', 'unknown')} (NF4 4-bit available)")


def load_for_verification():
    """Load the training model exactly as scripts/train.py will (same loader, same kwargs)."""
    bundle = load_model_and_tokenizer(
        MODEL_ID,
        torch_dtype=PRECISION,
        device="cuda",
        tuning_mode=CLI_TUNING_MODE,
        checkpoint=None,
        attn_implementation=cfg.model.attn_implementation,
        trust_remote_code=cfg.model.trust_remote_code,
        trainable=True,
        for_training=True,
        max_new_tokens=cfg.agent.max_new_tokens,
        quantization=cfg.training.quantization,
    )
    if bundle.model is None or bundle.tokenizer is None:
        raise RuntimeError("the loader returned no model/tokenizer")
    return bundle.model, bundle.tokenizer


raw_model, tokenizer = load_for_verification()
print(f"base loaded    : {MODEL_ID} | mode={cfg.training.tuning_mode} quant={cfg.training.quantization} "
      f"| dtype {PRECISION}")
print(f"architecture   : {type(raw_model).__name__} | "
      f"config.architectures={getattr(getattr(raw_model, 'config', None), 'architectures', None)}")
if CLI_TUNING_MODE == "qlora":
    _qcfg = getattr(getattr(raw_model, "config", None), "quantization_config", None)
    if not _qcfg:
        raise RuntimeError(
            "tuning_mode=qlora but the loaded model carries no quantization_config: the base is NOT "
            "4-bit, so this run would be LoRA mislabelled as QLoRA. Stop and inspect the loader."
        )
    print(f"quantization   : {_qcfg if isinstance(_qcfg, dict) else _qcfg.to_dict()}")
before = count_parameters(raw_model)

# Qwen's hybrid projections come from the model's own module names, never a Llama-shaped list.
discovered = discover_target_modules(raw_model)
if not discovered:
    raise RuntimeError(
        "LoRA target discovery found no attention/MLP projection. Do not hand-write "
        "q_proj/k_proj/... for a different architecture: extend "
        "lifemate.training.lora_finetuning.HYBRID_SUFFIXES or set lora.target_modules deliberately."
    )
print(f"discovered     : {len(discovered)} projection suffixes -> {', '.join(discovered)}")

if CLI_TUNING_MODE == "qlora":
    from lifemate.training.lora_finetuning import configure_qlora  # noqa: PLC0415

    lora_report = configure_qlora(raw_model, tokenizer, cfg)
else:
    from lifemate.training.lora_finetuning import configure_lora  # noqa: PLC0415

    lora_report = configure_lora(raw_model, cfg)
trainable_model = lora_report["model"]
if lora_report["trainable_params"] <= 0:
    raise RuntimeError("the adapter was attached but zero parameters are trainable: nothing would train")
print(f"adapter        : r={lora_report['r']} alpha={lora_report['alpha']} "
      f"targets={lora_report['n_target_modules']} tensors={lora_report['n_adapter_tensors']}")
print(f"trainable      : {lora_report['trainable_params']:,} / {lora_report['total_params']:,} "
      f"({lora_report['trainable_percentage']}%) grad_ckpt={lora_report['gradient_checkpointing']}")

# Real gradient-flow probe: forward + backward on one record, then inspect the adapter grads.
from lifemate.training.collator import SupervisedCollator, build_supervised_example, to_torch_batch  # noqa: E402

_probe_records = [
    json.loads(line)
    for line in (TRAIN_DATASET_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines()[:4]
]
_collator = SupervisedCollator.from_tokenizer(tokenizer)
_examples = [
    build_supervised_example(tokenizer, list(r["messages"]), str(r["target"]), cfg.training.max_seq_length)
    for r in _probe_records
]
_examples = [ex for ex in _examples if ex.get("kept")][:1]
if not _examples:
    raise RuntimeError("the first records cannot be tokenised at this sequence length; see section 5")
trainable_model.train()
_batch = to_torch_batch(_collator(_examples), "cuda")
with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
    _loss = trainable_model(**_batch).loss
_loss.backward()
_trainable = [(n, p) for n, p in trainable_model.named_parameters() if p.requires_grad]
_with_grad = [n for n, p in _trainable if p.grad is not None]
_nonzero = [n for n, p in _trainable if p.grad is not None and bool(p.grad.abs().sum() > 0)]
print(f"gradient probe : loss={float(_loss.detach()):.4f} | {len(_with_grad)}/{len(_trainable)} trainable "
      f"tensors got a grad, {len(_nonzero)} non-zero")
if not _nonzero:
    raise RuntimeError(
        "no non-zero adapter gradient after one backward pass. With gradient checkpointing a frozen "
        "base needs enable_input_require_grads(); configure_lora/configure_qlora call it, so check the "
        "peft version before spending GPU hours."
    )

MODEL_FACTS = {
    "model_id": MODEL_ID,
    "tuning_mode": cfg.training.tuning_mode,
    "quantization": cfg.training.quantization,
    "total_params": before["total"],
    "trainable_params": lora_report["trainable_params"],
    "trainable_pct": lora_report["trainable_percentage"],
    "targets": discovered,
    "gradient_checkpointing": lora_report["gradient_checkpointing"],
    "compute_dtype": str(COMPUTE_DTYPE).replace("torch.", ""),
    "peak_load_mib": int(torch.cuda.max_memory_allocated() // 2**20),
    "grad_tensors_nonzero": len(_nonzero),
}
print(f"peak during verification: {MODEL_FACTS['peak_load_mib']} MiB allocated")

# Release before the probe/real run: they load their own copy in a subprocess.
del trainable_model, raw_model, tokenizer, _collator, _examples, _trainable, _batch, _loss, _probe_records
free_gpu()


In [ ]:
# --- 6c. Bind the CLI contract: flags, choices, resume form ----------------
TRAIN_FLAGS = require_flags(
    "train.py",
    ["--config", "--dataset-dir", "--model", "--output-dir", "--seed", "--max-steps", "--tuning", "--run-name"],
    hint="These are the flags section 8 drives the trainer with.",
)

if RUN["quantized_base"] and not QLORA_SUPPORTED:
    # The 4-bit stack itself was verified above; what is missing is the trainer's own quantized
    # weight-loading path. Launching anyway would train a bf16/fp16 base and silently mislabel the
    # run, so stop here with the exact options.
    raise RuntimeError(
        f"TRAINING_METHOD=qlora requested, but this checkout's train.py accepts only {TUNING_CHOICES}. "
        "The NF4 + prepare_model_for_kbit_training stack above is verified (bitsandbytes imported, "
        "adapter gradients flowed), but scripts/train.py owns weight loading and has no quantized "
        "branch, so the notebook cannot inject 4-bit loading into the subprocess it must use.\n"
        "Options:\n"
        "  1. TRAINING_METHOD=lora (works today; on a T4 use MAX_SEQ_LENGTH<=4096), or\n"
        "  2. pin BRANCH_OR_COMMIT to a commit that ships `--tuning qlora`.\n"
        "Required repo change: a `--tuning qlora` value + BitsAndBytesConfig in the loader path."
    )


def resolve_flag(script_name, candidates, meaning):
    """First existing spelling of a flag, so the notebook survives a rename."""
    have = script_flags(script_name)
    for name in candidates:
        if name in have:
            return name
    raise RuntimeError(
        f"{script_name} exposes none of {candidates} (needed for the {meaning}). It accepts: "
        f"{', '.join(sorted(have))}. Update the notebook's candidate list to match the checkout."
    )


SEQ_FLAG = resolve_flag("train.py", ["--max-seq-length", "--seq-len"], "sequence budget")
RESUME_FLAG = resolve_flag("train.py", ["--resume-from", "--resume"], "checkpoint resume")
print(f"seq flag: {SEQ_FLAG} | resume flag: {RESUME_FLAG}")


def train_args(*, max_steps=None, resume=None, output_dir=None):
    """The exact argv section 7/8/9 use; one builder so probe and real run cannot drift."""
    args = [
        "--config", CONFIG_PATH,
        "--dataset-dir", TRAIN_DATASET_DIR,
        "--model", MODEL_ID,
        "--output-dir", output_dir or CKPT_OUTPUT_DIR,
        "--run-name", RUN_NAME,
        "--tuning", CLI_TUNING_MODE,
        "--seed", int(SEED),
        SEQ_FLAG, int(RUN["max_seq_length"]),
    ]
    if max_steps:
        args += ["--max-steps", int(max_steps)]
    if resume:
        args += [RESUME_FLAG, str(resume)]
    return args


print("train argv (no max-steps):", " ".join(str(a) for a in train_args()))


## 7. Memory preflight: estimate, then measure

Purpose: two independent numbers before spending GPU hours.

1. **Estimate** from the repo's `run_memory_preflight` (labelled as a heuristic projection, not a
   measurement) — weights, gradients, optimizer state, activations, total.
2. **Measurement** from a real 2-step training run of the real trainer on the real model, sampled
   from `nvidia-smi` while the child process runs. That is the only number that can be trusted.

Refusal rule: an estimate that cannot fit the card stops the notebook here, and a measured peak
above 92% of free VRAM stops it before the long run. Cost: 1-2 optimizer steps ≈ 1-4 min GPU; the
checkpoints it writes are deleted immediately (`--max-steps 2` → `step-1`, `step-2`).


In [ ]:
# --- 7a. Heuristic estimate + explicit refusal ------------------------------
from lifemate.training.memory_preflight import format_report, run_memory_preflight  # noqa: E402

ESTIMATE = run_memory_preflight(param_count=MODEL_FACTS["total_params"], cfg=cfg,
                                seq_len=cfg.training.max_seq_length)
print(format_report(ESTIMATE))
print(
    f"\nreality check: the repo estimate sizes the activation term from a heuristic, so it is an "
    f"ESTIMATE. TRAINABLE params are measured ({MODEL_FACTS['trainable_params']:,}); "
    f"weights are measured at load time ({MODEL_FACTS['peak_load_mib']} MiB peak allocated)."
)

if ESTIMATE["verdict"] == "does_not_fit":
    raise RuntimeError(
        f"the estimate ({ESTIMATE['total_gb']} GB) exceeds the {GPU['vram_total_gb']} GB on "
        f"{GPU['name']}. Lower MAX_SEQ_LENGTH (currently {cfg.training.max_seq_length}) and/or "
        f"MICRO_BATCH_SIZE, raise GRADIENT_ACCUMULATION_STEPS to keep the effective batch, or pick a "
        f"bigger runtime. Suggestions: {ESTIMATE['suggestions']}"
    )
if ESTIMATE["verdict"] == "tight":
    print("warning: the estimate is 'tight' — the measured probe below decides, not this number")


In [ ]:
# --- 7b. Measured preflight: one 2-step run of the real trainer ------------
PROBE_DIR = LOCAL_OUTPUT_ROOT / "probe"
if PROBE_DIR.exists():
    shutil.rmtree(PROBE_DIR)

print("running a 2-step probe with the real trainer (this loads the weights again, on purpose):")
run_cli("train.py", train_args(max_steps=2, output_dir=PROBE_DIR), sample_peak=True, peak_interval=2.0)

_peak_mib = LAST_CLI_PEAK_MIB
PROBE = {
    "sampled_peak_mib": _peak_mib,
    "sampled_peak_gb": None if _peak_mib is None else round(_peak_mib * 2**20 / 1e9, 2),
    "device_total_gb": GPU.get("vram_total_gb"),
    "estimate_total_gb": ESTIMATE["total_gb"],
    "estimate_verdict": ESTIMATE["verdict"],
    "probe_output": str(PROBE_DIR),
}
if _peak_mib is None:
    raise RuntimeError(
        "could not sample device memory during the probe (nvidia-smi unavailable). The probe's own "
        "torch peak line in the output above is the number to use; refusing to guess."
    )
HEADROOM_FRACTION = 0.92
_used_fraction = PROBE["sampled_peak_gb"] / float(GPU["vram_total_gb"])
PROBE["used_fraction_of_total"] = round(_used_fraction, 4)
print(json.dumps(PROBE, indent=2))
if _used_fraction > HEADROOM_FRACTION:
    raise RuntimeError(
        f"the probe peaked at {PROBE['sampled_peak_gb']} GB = {_used_fraction:.0%} of "
        f"{GPU['vram_total_gb']} GB. A full epoch will fragment past that and OOM. Lower "
        "MAX_SEQ_LENGTH (halving it cuts the activation term roughly in half) or MICRO_BATCH_SIZE=1 "
        "with a larger GRADIENT_ACCUMULATION_STEPS, then re-run this section."
    )
print(f"probe fits: {PROBE['sampled_peak_gb']} GB peak of {GPU['vram_total_gb']} GB "
      f"({_used_fraction:.0%}) — real measurement, not the heuristic")

# The probe checkpoints are throwaway: keeping them would collide with the real run's step numbering.
shutil.rmtree(PROBE_DIR, ignore_errors=True)
free_gpu()


## 8. Training

Purpose: run the repo's trainer (its loop, its gradient accumulation, checkpointing, LR schedule
and validation) for the configured epochs. The notebook only assembles argv and watches the output.

While it runs, this cell **syncs each completed checkpoint to Drive** as soon as it appears
(`checkpoint_ready()` proves `training_state.json`, weights and `optimizer_state.pt` are all
present), through a staging directory + rename + marker. That is what makes a disconnect survivable.

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` (exported in section 1) is inherited by the
trainer subprocess and measurably reduces allocator fragmentation at long sequence lengths. It
changes no number in the run, only how the driver hands memory back.

Cost: the dominant term. Rough ESTIMATE for 4368 records, 3 epochs, effective batch 8,
~1650 steps: T4 16 GB ≈ 4-7 h; L4 ≈ 2.5-4 h; A100-40/80 ≈ 1-2 h. A Colab runtime that idles while
you read logs bills exactly the same — stop it when done (section 13).


In [ ]:
# --- 8. Train, streaming output and mirroring finished checkpoints --------
SYNCED_CHECKPOINTS = {}
TRAIN_LOG_PATH = LOCAL_OUTPUT_ROOT / "train_stdout.log"


def latest_step():
    """Highest step number present locally, or 0."""
    steps = [int(p.name.split("-")[-1]) for p in RUN_DIR.glob("step-*")] if RUN_DIR.is_dir() else []
    return max(steps) if steps else 0


def sync_finished_checkpoints(verbose=True):
    """Copy every complete, not-yet-synced step dir to Drive (atomic + marker)."""
    run_root = RUN_DIR
    if not run_root.is_dir():
        return []
    done = []
    for step_dir in sorted(run_root.glob("step-*"), key=lambda p: int(p.name.split("-")[-1])):
        if step_dir.name in SYNCED_CHECKPOINTS:
            continue
        ok, reason = checkpoint_ready(step_dir)
        if not ok:
            if verbose:
                print(f"  [sync] {step_dir.name} not ready: {reason}")
            continue
        if DRIVE_ROOT is not None:
            target = DRIVE_PATHS["checkpoints"] / RUN_NAME / step_dir.name
            atomic_sync(step_dir, target, extra={"step": int(step_dir.name.split("-")[-1]),
                                                 "run_name": RUN_NAME, "model_id": MODEL_ID})
        SYNCED_CHECKPOINTS[step_dir.name] = str(time.strftime("%H:%M:%S"))
        done.append(step_dir.name)
        if verbose:
            print(f"  [sync] {step_dir.name} durable ({SYNCED_CHECKPOINTS[step_dir.name]})")
    return done


def run_training(*, max_steps=None, resume=None):
    """Run the trainer in a subprocess; mirror checkpoints; raise on a non-zero exit."""
    global LAST_CLI_PEAK_MIB
    env = dict(os.environ)
    # 1a exported expandable_segments already; restated so a re-run of this cell alone keeps it.
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("PYTHONUNBUFFERED", "1")
    env.setdefault("WANDB_MODE", "offline")   # only used if training.report_to asks for wandb
    cmd = [sys.executable, str(REPO_DIR / "scripts" / "train.py"),
           *[str(a) for a in train_args(max_steps=max_steps, resume=resume)]]
    def _tick():
        # Live progress: finished checkpoints (mirrored immediately) and the step counter.
        before = len(SYNCED_CHECKPOINTS)
        sync_finished_checkpoints(verbose=False)
        if len(SYNCED_CHECKPOINTS) != before:
            print(f"    [run] step {latest_step()} checkpoint ready "
                  f"({len(SYNCED_CHECKPOINTS)} synced)", flush=True)

    rc, lines = stream_process(cmd, env, sample_peak=True, peak_interval=15, status_prefix="run",
                               on_tick=_tick)
    TRAIN_LOG_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
    sync_finished_checkpoints()
    if rc != 0:
        raise RuntimeError(
            f"train.py exited {rc}. Full output saved to {TRAIN_LOG_PATH}. 130 means it was "
            "interrupted after writing an interrupt-<step> checkpoint — resume with section 9."
        )
    return lines


# A Drive checkpoint is only trusted with its completion marker; a local one is trusted directly.
resume_target = None
if str(RESUME_FROM_CHECKPOINT).strip():
    resume_target = Path(str(RESUME_FROM_CHECKPOINT).strip())
    ok, why = checkpoint_ready(resume_target, require_marker=DRIVE_ROOT is not None and str(resume_target).startswith(str(DRIVE_ROOT)))
    if not ok:
        raise RuntimeError(f"RESUME_FROM_CHECKPOINT is unusable: {why}")
    print(f"resuming from {resume_target}")
else:
    print("fresh run (set RESUME_FROM_CHECKPOINT in section 2 to continue an interrupted one)")

if resume_target is None and RUN_DIR.is_dir() and any(RUN_DIR.glob("step-*")):
    _existing = sorted(RUN_DIR.glob("step-*"), key=lambda p: int(p.name.split("-")[-1]))
    raise RuntimeError(
        f"{RUN_DIR} already holds {len(_existing)} checkpoint(s) (latest {_existing[-1].name}). "
        "The repo's CheckpointManager refuses to overwrite a populated step directory, and starting "
        "a second fresh run in the same RUN_NAME would mix two runs' steps. Do one of:\n"
        f"  - resume: set RESUME_FROM_CHECKPOINT = '{_existing[-1]}' in section 2 and re-run this cell;\n"
        "  - or change RUN_NAME in section 2 for a new experiment.\n"
        f"Nothing was launched. Existing dirs stay untouched."
    )

print("CU cost is billed for wall-clock GPU time; a slow epoch is a bigger bill.")
train_output = run_training(max_steps=int(MAX_STEPS) or None, resume=resume_target)

summary_path = Path(cfg.training.output_dir) / "train_summary.json"
if summary_path.is_file():
    TRAIN_SUMMARY = json.loads(summary_path.read_text(encoding="utf-8"))
    _history = (TRAIN_SUMMARY.get("result") or {}).get("metrics_history") or []
    print("\nstep   loss      val_loss  lr")
    for row in _history:
        loss, val = row.get("train_loss"), row.get("val_loss")
        lr = row.get("learning_rate")
        print(f"{str(row.get('step')):<6} "
              f"{(f'{loss:.4f}' if isinstance(loss, float) else '-'):<9} "
              f"{(f'{val:.4f}' if isinstance(val, float) else '-'):<9} "
              f"{(f'{lr:.3e}' if isinstance(lr, float) else '-')}")
    print(f"\nsteps={TRAIN_SUMMARY.get('result', {}).get('steps')} "
          f"epochs={TRAIN_SUMMARY.get('result', {}).get('epochs')} "
          f"final_loss={TRAIN_SUMMARY.get('result', {}).get('final_loss')} "
          f"val_loss={TRAIN_SUMMARY.get('result', {}).get('val_loss')}")
    print(f"tuning_mode={TRAIN_SUMMARY.get('tuning_mode')} model={TRAIN_SUMMARY.get('model')}")
else:
    TRAIN_SUMMARY = None
    print(f"warning: {summary_path} missing; the run's stdout is in {TRAIN_LOG_PATH}")
LAST_STEP_DIR = max((p for p in RUN_DIR.glob("step-*")), key=lambda p: int(p.name.split("-")[-1])) if RUN_DIR.is_dir() else None
print(f"latest checkpoint: {LAST_STEP_DIR}")


## 9. Checkpoint contents + a demonstrated resume

Purpose: show that the saved directory is enough to *continue the same optimisation*, and prove it
by actually resuming for two more steps. A checkpoint from this trainer holds:

`adapter_model.safetensors` + `adapter_config.json` (or full weights), the tokenizer,
`training_state.json` (step, epoch, seed, `config_hash`, dataset manifest hash, is_lora, precision,
torch/CUDA/python RNG state), `optimizer_state.pt` (AdamW moments + scheduler), `lifemate_meta.json`
(model id, tuning mode, config), and — after a Drive sync — `SYNC_COMPLETE.json`.

Loading only `adapter_model.safetensors` is **adapter-only loading**; it restores weights and nothing
about the optimizer. This notebook refuses to call that a resume: `checkpoint_ready()` requires
`training_state.json` **and** `optimizer_state.pt`, and the trainer itself refuses a config-hash or
dataset-hash mismatch.

Cost: 2 optimizer steps ≈ 1-3 min.


In [ ]:
# --- 9a. Inspect the checkpoint, then prove continuation from the right step ---
if LAST_STEP_DIR is None:
    raise RuntimeError(f"no step-* checkpoint under {RUN_DIR}; run section 8 first")

state = json.loads((LAST_STEP_DIR / "training_state.json").read_text(encoding="utf-8"))
meta = json.loads((LAST_STEP_DIR / "lifemate_meta.json").read_text(encoding="utf-8"))
CONTENTS = {
    "dir": str(LAST_STEP_DIR),
    "step": state.get("step"),
    "epoch": state.get("epoch"),
    "seed": state.get("seed"),
    "config_hash": state.get("config_hash"),
    "dataset_manifest_hash": state.get("dataset_manifest_hash"),
    "is_lora": state.get("is_lora"),
    "precision": state.get("precision"),
    "rng_state_saved": {"torch": state.get("torch_rng_state") is not None,
                        "cuda": state.get("cuda_rng_state") is not None,
                        "python": state.get("python_rng_state") is not None},
    "optimizer_state_present": (LAST_STEP_DIR / "optimizer_state.pt").is_file(),
    "model_identity": meta.get("model_name"),
    "tuning_mode": meta.get("tuning_mode"),
    "files": sorted(p.name for p in LAST_STEP_DIR.iterdir()),
    "size_mb": round(sum(p.stat().st_size for p in LAST_STEP_DIR.rglob("*") if p.is_file()) / 2**20, 1),
}
print(json.dumps(CONTENTS, indent=2, ensure_ascii=False))

ok, why = checkpoint_ready(LAST_STEP_DIR)
if not ok:
    raise RuntimeError(f"the checkpoint is not resumable: {why}")
if CONTENTS["config_hash"] != CONFIG_HASH:
    raise RuntimeError(
        f"the checkpoint was trained under config hash {CONTENTS['config_hash']}, this session's "
        f"config hashes to {CONFIG_HASH}. Something changed between the run and now (lr, seed, seq "
        "len, batch, epochs, precision, tuning mode); the trainer would refuse it for the same reason."
    )

RESUME_PROOF = {"from_step": int(CONTENTS["step"]), "to_step": None, "continued": False}
_resume_limit = int(CONTENTS["step"]) + 2
resume_log = run_training(max_steps=_resume_limit, resume=LAST_STEP_DIR)
resumed_lines = [line for line in resume_log if line.startswith("resumed")]
if not resumed_lines:
    raise RuntimeError(
        f"the trainer did not print a `resumed` line; output saved to {TRAIN_LOG_PATH}. "
        "That line is the only evidence that step continuity was restored."
    )
print(resumed_lines[0])
_new_dirs = sorted((p for p in RUN_DIR.glob("step-*") if int(p.name.split("-")[-1]) > int(CONTENTS["step"])),
                   key=lambda p: int(p.name.split("-")[-1]))
if not _new_dirs:
    raise RuntimeError(
        f"no step dir above {CONTENTS['step']} appeared: the run did not continue. Check that "
        "MAX_STEPS / NUM_EPOCHS leave room after the checkpoint's step (total steps must exceed it)."
    )
RESUME_PROOF["to_step"] = int(_new_dirs[-1].name.split("-")[-1])
RESUME_PROOF["continued"] = RESUME_PROOF["to_step"] > RESUME_PROOF["from_step"]
RESUME_PROOF["synced"] = sorted(SYNCED_CHECKPOINTS)
if DRIVE_ROOT is not None:
    (LOCAL_OUTPUT_ROOT / "resume_proof.json").write_text(json.dumps(RESUME_PROOF, indent=2), encoding="utf-8")
    atomic_sync(_new_dirs[-1], DRIVE_PATHS["checkpoints"] / RUN_NAME / _new_dirs[-1].name,
                extra={"step": RESUME_PROOF["to_step"], "resume_proof": True})
print(json.dumps(RESUME_PROOF, indent=2))
if not RESUME_PROOF["continued"]:
    raise RuntimeError("resume did not advance the step counter; this is not a valid resume")
LAST_STEP_DIR = _new_dirs[-1]
free_gpu()


## 10. Evaluation

Purpose: score the trained checkpoint with the repo's evaluator. Teacher-forced metrics (validation
loss, tool-name accuracy, argument accuracy, action exact match) are available whenever the model
loads; the closed loop needs the organizers' `zaic_tools` SDK and is attempted only when that import
succeeds.

**Compliance.** Teacher-forced numbers on the held-out split of `train` are an *offline* signal.
Mock-environment numbers are printed with `MOCK ENV / SYNTHETIC` and are never presented as an
official score; nothing here writes a submission file.

Cost: validation-only over 482 records ≈ 3-8 min on T4; a 20-sample closed loop adds a few minutes.


In [ ]:

# --- 10. Evaluate: teacher-forced (always), closed loop (only if possible) --
require_flags("evaluate.py", ["--validation-only", "--closed-loop", "--checkpoint", "--model-mode",
                              "--config", "--predictions", "--output", "--limit", "--split"])


def eval_args(output_dir, *, validation_only=True, closed_loop=False, limit=None):
    """argv for the evaluator; teacher-forced scoring uses the val decision records."""
    args = [
        "--config", CONFIG_PATH,
        "--checkpoint", LAST_STEP_DIR,
        "--model-mode", "lora",
        "--split", "train",
        "--output", output_dir,
    ]
    if validation_only:
        args += ["--validation-only", "--predictions", TRAIN_DATASET_DIR / "val.jsonl"]
    if closed_loop:
        args += ["--closed-loop"]
        if limit:
            args += ["--limit", int(limit)]
    return args


EVAL_DIR = LOCAL_OUTPUT_ROOT / "evaluation"
if EVAL_DIR.exists():
    shutil.rmtree(EVAL_DIR)

run_cli("evaluate.py", eval_args(EVAL_DIR))
metrics_path = EVAL_DIR / "metrics.json"
if not metrics_path.is_file():
    raise RuntimeError(f"evaluate.py wrote no metrics.json under {EVAL_DIR}; see its output above")
EVAL_PAYLOAD = json.loads(metrics_path.read_text(encoding="utf-8"))
TEACHER_FORCED = {
    "n_records": (EVAL_PAYLOAD.get("teacher_forced") or {}).get("n_records"),
    "metrics": EVAL_PAYLOAD.get("metrics"),
    "mock_env": EVAL_PAYLOAD.get("mock_env"),
    "provenance": EVAL_PAYLOAD.get("provenance"),
}
print("teacher-forced metrics — offline, on the held-out split of `train`. NOT a leaderboard score:")
print(json.dumps(TEACHER_FORCED["metrics"], indent=2)[:3000])
if EVAL_PAYLOAD.get("mock_env"):
    raise RuntimeError(
        "this report is marked MOCK ENV / SYNTHETIC; something routed evaluation through the mock "
        "environment. It must never be quoted as a score."
    )

# Closed loop: only with the organizers' SDK present. Never silently upgraded to the mock.
CLOSED_LOOP = {"attempted": False, "reason": "data/train/zaic_tools is absent"}
if (DATA_ROOT / "train" / "zaic_tools").is_dir():
    CLOSED_LOOP["attempted"] = True
    CLOSED_DIR = LOCAL_OUTPUT_ROOT / "evaluation_closed_loop"
    if CLOSED_DIR.exists():
        shutil.rmtree(CLOSED_DIR)
    try:
        run_cli("evaluate.py", eval_args(CLOSED_DIR, validation_only=False, closed_loop=True, limit=20),
                sample_peak=True)
        CLOSED_LOOP["result"] = "ran against the official environment, train split, first 20 samples"
        _closed = json.loads((CLOSED_DIR / "metrics.json").read_text(encoding="utf-8"))
        CLOSED_LOOP["status_counts"] = ((_closed.get("closed_loop") or {}).get("status_counts"))
        CLOSED_LOOP["mock_env"] = _closed.get("mock_env")
    except RuntimeError as exc:
        CLOSED_LOOP["result"] = f"not run: {exc}"
else:
    print(
        "closed loop skipped: data/train/zaic_tools is absent, and the notebook will not substitute "
        "the mock for it. With the SDK present, the command is:\n"
        f"  python scripts/evaluate.py --closed-loop --split train --limit 20 --model-mode lora "
        f"--checkpoint {LAST_STEP_DIR} --config {CONFIG_PATH} --output {LOCAL_OUTPUT_ROOT}/evaluation_closed_loop\n"
        "Without it, the only closed-loop option is the mock, whose every artifact is stamped "
        "MOCK ENV / SYNTHETIC and which is NEVER a score:\n"
        f"  python scripts/evaluate.py --closed-loop --use-mock-env --split train --limit 5 "
        f"--model-mode lora --checkpoint {LAST_STEP_DIR} --output {LOCAL_OUTPUT_ROOT}/evaluation_mock"
    )
print(json.dumps(CLOSED_LOOP, indent=2))
if DRIVE_ROOT is not None and EVAL_DIR.is_dir():
    atomic_sync(EVAL_DIR, DRIVE_PATHS["outputs"] / f"evaluation_{RUN_NAME}", extra={"kind": "evaluation"})
free_gpu()


## 11. Export

Purpose: produce a portable artifact. The default is the **adapter** — a few tens of MB, loadable at
inference with `--model-mode lora --checkpoint <dir>`, no merge required. A merge into full weights
is available when the GPU can hold the fp16 base during the merge; it is off by default because it
costs VRAM, disk and time for no training benefit.

Exported: adapter weights, tokenizer, `export_meta.json` (reproducibility: model id, config hash,
dataset hash, git commit, libraries, step) and the effective YAML. Then synced to Drive.

Cost: <1 min for the adapter copy; a merge on a 4B model ≈ 2-6 min plus ~9 GB of disk.


In [ ]:
# --- 11. Export the adapter (+ optional merge) ----------------------------
require_flags("export_model.py", ["--ckpt", "--output", "--merge-lora", "--base-model", "--dtype", "--config"])
MERGE_LORA = False  # set True to also produce merged fp16 weights (needs ~20 GB free VRAM+disk)

# Re-read the state: section 9 may have resumed past the checkpoint inspected there.
EXPORT_STEP = int(json.loads((LAST_STEP_DIR / "training_state.json").read_text(encoding="utf-8"))["step"])
EXPORT_DIR = LOCAL_OUTPUT_ROOT / "export" / f"{RUN_NAME}-step{EXPORT_STEP}"
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
ADAPTER_DIR = EXPORT_DIR / "adapter"
ADAPTER_DIR.mkdir(parents=True)

# Optimizer state is not part of a portable adapter; everything else in the step dir is.
KEEP = ("adapter_model.safetensors", "adapter_model.bin", "adapter_config.json", "tokenizer.json",
        "tokenizer_config.json", "special_tokens_map.json", "vocab.json", "merges.txt",
        "generation_config.json", "chat_template.jinja", "config.json")
copied = []
for name in KEEP:
    src = LAST_STEP_DIR / name
    if src.is_file():
        shutil.copy2(src, ADAPTER_DIR / name)
        copied.append(name)
if not any(name.startswith("adapter_model") for name in copied):
    raise RuntimeError(
        f"{LAST_STEP_DIR} holds no adapter weights (found: {sorted(p.name for p in LAST_STEP_DIR.iterdir())}). "
        "This notebook only exports LoRA/QLoRA adapters; a full-finetune checkpoint would need "
        "scripts/export_model.py without the adapter copy."
    )
shutil.copy2(CONFIG_PATH, EXPORT_DIR / "effective_config.yaml")
_export_meta = {
    "model_id": MODEL_ID,
    "base_model": MODEL_ID,
    "kind": "lora_adapter",
    "source_checkpoint": str(LAST_STEP_DIR),
    "step": EXPORT_STEP,
    "training_state": json.loads((LAST_STEP_DIR / "training_state.json").read_text(encoding="utf-8")),
    "config_hash": CONFIG_HASH,
    "dataset_manifest_hash": DATASET_REPORT["split_manifest"]["group_hash"],
    "dataset": DATASET_REPORT,
    "git_commit": subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR),
                                 capture_output=True, text=True).stdout.strip(),
    "libraries": {"torch": torch.__version__, "transformers": transformers.__version__,
                  "peft": peft.__version__, "python": sys.version.split()[0]},
    "precision": PRECISION,
    "trainable_params": MODEL_FACTS["trainable_params"],
    "files": copied,
    "how_to_run": (f"python scripts/infer.py --model-mode lora --checkpoint {ADAPTER_DIR} "
                   f"--model {MODEL_ID} --split public_test --data-root ./data "
                   "--output submission.jsonl"),
    "caveat": "An adapter reproduces the weights; it does not carry optimizer state. For resuming "
              "training use the step-<n> directory, not this export.",
}
(EXPORT_DIR / "export_meta.json").write_text(json.dumps(_export_meta, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"exported adapter -> {ADAPTER_DIR} ({sum((ADAPTER_DIR / n).stat().st_size for n in copied) / 2**20:.1f} MB)")

if MERGE_LORA:
    _free_gb = float(GPU.get("vram_free_gb") or 0)
    if _free_gb < 20:
        print(f"merge skipped: {_free_gb} GB free VRAM, a 4B fp16 merge needs ~20 GB. The adapter is "
              "the deliverable; the repo merges it at load time when you pass --model-mode lora.")
    else:
        run_cli("export_model.py", ["--ckpt", LAST_STEP_DIR, "--output", EXPORT_DIR / "merged",
                                    "--merge-lora", "--base-model", MODEL_ID,
                                    "--config", CONFIG_PATH, "--dtype", PRECISION,
                                    "--force"], sample_peak=True)
        print("merged weights written; verify them with scripts/infer.py --model-mode full")

if DRIVE_ROOT is not None:
    atomic_sync(EXPORT_DIR, DRIVE_PATHS["outputs"] / EXPORT_DIR.name, extra={"kind": "export"})
    print(f"on Drive: {DRIVE_PATHS['outputs'] / EXPORT_DIR.name}")
print("download it from Drive, or copy to your machine, then:")
print(f"  python scripts/infer.py --model-mode lora --checkpoint <downloaded>/adapter \\\n"
      f"      --split public_test --data-root ./data --output submission.jsonl \\\n"
      f"      --model {MODEL_ID}")


## 12. Inference smoke test

Purpose: load the exported adapter through the repo's loader, generate a real completion, and push
it through the agent's parser + validator + one agent step. This catches adapter/tokenizer/chat-
template mismatch — the failure mode that makes "the model loaded fine" worthless.

**A successful load is not a successful tool-calling evaluation.** This cell proves plumbing only.

Cost: ~30-60 s on GPU for one short generation.


In [ ]:
# --- 12. Load the export, generate, parse, validate, one agent step --------
from lifemate.agent.tool_parser import parse_model_output
from lifemate.agent.tool_registry import ToolRegistry
from lifemate.data.catalog import ToolCatalog
from lifemate.data.loader import load_samples
from lifemate.env.factory import create_environment
from lifemate.models.generation import HFGenerator
from lifemate.models.loader import load_model_and_tokenizer

bundle = load_model_and_tokenizer(
    MODEL_ID,
    torch_dtype=PRECISION,
    device="cuda",
    tuning_mode="lora",
    checkpoint=ADAPTER_DIR,
    attn_implementation=cfg.model.attn_implementation,
    trust_remote_code=cfg.model.trust_remote_code,
    max_new_tokens=cfg.agent.max_new_tokens,
)
SMOKE = {"adapter": str(ADAPTER_DIR), "model_mode": bundle.model_mode}
print(f"loaded adapter : {bundle.model_mode} | "
      f"{bundle.meta['n_parameters']['total']:,} params | dtype {bundle.meta['torch_dtype']}")

generator = HFGenerator(bundle.model, bundle.tokenizer, max_new_tokens=cfg.agent.max_new_tokens,
                        temperature=0.0, device=bundle.device)
catalog = ToolCatalog.from_file(DATA_ROOT / "train" / "train_tools_catalog.json")

_val_records = [json.loads(line) for line in (TRAIN_DATASET_DIR / "val.jsonl").read_text(encoding="utf-8").splitlines()[:5]]
_record = _val_records[0]
SMOKE["sample_id"] = _record.get("sample_id")
SMOKE["step_index"] = _record.get("step_index")
SMOKE["gold_target"] = _record.get("target")

# The record's own messages are the exact prompt bytes training used.
raw_output = generator.generate(list(_record["messages"]))
SMOKE["raw_output"] = raw_output
print(f"generated      : {raw_output!r}")

registry = ToolRegistry(catalog)                      # every app; a real sample narrows this
env = create_environment("train", str(_record.get("sample_id") or "smoke"), DATA_ROOT,
                         use_mock=True, catalog=catalog)   # MOCK ENV: plumbing only
action = parse_model_output(raw_output, registry, env)
SMOKE["action"] = {"type": action.type, "name": action.name, "arguments": action.arguments,
                   "error": action.error}
print(f"parsed+validated: {json.dumps(SMOKE['action'], ensure_ascii=False)}")
if action.type == "invalid":
    print("note: the model produced an invalid action — that is a training-signal question, not a "
          "plumbing bug. The parser refusing it is the correct behaviour.")

# Agent path: one real episode on a real labelled sample, with the mock tool backend.
samples = {str(s.id): s for s in load_samples(DATA_ROOT, "train")}
sample = samples.get(str(_record.get("sample_id")))
if sample is None:
    print(f"warning: sample {_record.get('sample_id')} not in the split; skipping the agent step")
else:
    from lifemate.agent.agent import ToolAgent
    agent_env = create_environment("train", str(sample.id), DATA_ROOT, use_mock=True, catalog=catalog)
    agent = ToolAgent(generator=generator, registry=ToolRegistry(catalog).for_sample(sample),
                      env=agent_env, config=cfg, max_steps=cfg.agent.max_steps)
    result = agent.run(sample, transcript=sample.audio_text)
    SMOKE["agent"] = {"status": result.status, "finish_reason": result.finish_reason,
                      "n_llm_calls": result.n_llm_calls, "tool_calls": result.tool_calls,
                      "mock_env": True}
    print("MOCK ENV / SYNTHETIC agent episode (plumbing only, not a score):")
    print(json.dumps(SMOKE["agent"], indent=2, ensure_ascii=False)[:2000])

if "agent" not in SMOKE:
    SMOKE["agent"] = {"status": "skipped", "reason": "sample unavailable"}
SMOKE["verdict"] = ("adapter loaded and generated, output parsed by the repo parser; this is NOT a "
                    "tool-calling accuracy measurement")
print(SMOKE["verdict"])
if DRIVE_ROOT is not None:
    (LOCAL_OUTPUT_ROOT / "smoke_test.json").write_text(
        json.dumps(SMOKE, indent=2, ensure_ascii=False), encoding="utf-8")

# Nothing downstream needs the loaded model; keeping it alive only holds VRAM.
_gone = [name for name in ("generator", "bundle", "agent", "agent_env") if name in globals()]
for _name in _gone:
    globals()[_name] = None
free_gpu()
print("generation model released")



## 13. Resource management, final sync, shutdown

Purpose: leave the session in a clean, resumable state and stop paying. An allocated Colab runtime
bills Compute Units while it does nothing, so the last cell syncs results and releases memory;
then **disconnect the runtime yourself** (`Runtime -> Disconnect and delete runtime`).

Cheap vs expensive cells, and why the order is fixed:

* cheap (no GPU work): sections 1-5, 11, 13 — safe to re-run any time;
* GPU-heavy: 6 (weights), 7 (probe), 8 (training), 9 (2-step resume), 10 (evaluation), 12 (generation);
* weights are downloaded once into `HF_HOME` under the runtime-local work dir — re-running 6 does
  not re-download unless the runtime was recycled.

Cost: seconds of GPU (freeing) — and it stops the meter on everything else.


In [ ]:

# --- 13. Final sync + free everything ------------------------------------
# Report whatever exists: a section that failed earlier must not make clean-up crash too.
def _g(name, default=None):
    """Value of a notebook global, or ``default`` when that section never ran."""
    return globals().get(name, default)


FINAL = {
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "method": TRAINING_METHOD,
    "config_hash": (_g("CONTENTS") or {}).get("config_hash"),
    "steps_synced": sorted(_g("SYNCED_CHECKPOINTS") or {}),
}

if DRIVE_ROOT is not None:
    _report = {
        "run_config": _g("RUN"),
        "dataset": _g("DATASET_REPORT"),
        "model_facts": _g("MODEL_FACTS"),
        "memory": {"estimate": _g("ESTIMATE"), "probe": _g("PROBE")},
        "training_summary": _g("TRAIN_SUMMARY"),
        "teacher_forced": _g("TEACHER_FORCED"),
        "closed_loop": _g("CLOSED_LOOP"),
        "resume_proof": _g("RESUME_PROOF"),
        "smoke_test": _g("SMOKE"),
        "git_commit": (_g("_export_meta") or {}).get("git_commit"),
        "finished_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    (LOCAL_OUTPUT_ROOT / "run_report.json").write_text(
        json.dumps(_report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

    for _sub, _label in (("configs", f"configs_{RUN_NAME}"), ("export", "exports"),
                         ("evaluation", f"evaluation_{RUN_NAME}")):
        _src_dir = LOCAL_OUTPUT_ROOT / _sub
        if _src_dir.is_dir():
            atomic_sync(_src_dir, DRIVE_PATHS["outputs"] / _label, extra={"kind": _sub})
    _log_dir = DRIVE_PATHS["logs"] / RUN_NAME
    _log_dir.mkdir(parents=True, exist_ok=True)
    for _name in ("train_stdout.log", "run_report.json", "resume_proof.json", "smoke_test.json"):
        _src = LOCAL_OUTPUT_ROOT / _name
        if _src.is_file():
            shutil.copy2(_src, _log_dir / _name)
    with open(_log_dir / "train_stdout.log", "ab") as _handle:  # append-never-truncate + fsync
        _handle.flush()
        os.fsync(_handle.fileno())
    FINAL["drive"] = {
        "checkpoints": str(DRIVE_PATHS["checkpoints"] / RUN_NAME),
        "exports": str(DRIVE_PATHS["outputs"] / "exports"),
        "evaluation": str(DRIVE_PATHS["outputs"] / f"evaluation_{RUN_NAME}"),
        "logs": str(_log_dir),
    }
    print("on Drive:")
    for _label, _path in FINAL["drive"].items():
        print(f"  {_label:<12}: {_path}")

# Local step dirs are redundant once every one of them is durably on Drive.
_run_dir = _g("RUN_DIR")
_synced = _g("SYNCED_CHECKPOINTS") or {}
_local_steps = [p.name for p in _run_dir.glob("step-*")] if _run_dir is not None and _run_dir.is_dir() else []
_all_synced = bool(_local_steps) and all(name in _synced for name in _local_steps)
if DRIVE_ROOT is not None and _all_synced:
    for _stale in _run_dir.glob("step-*"):
        shutil.rmtree(_stale, ignore_errors=True)
    print("local checkpoints removed; the Drive copy + SYNC_COMPLETE.json is the durable artifact")
elif _local_steps:
    print(f"kept local checkpoints under {_run_dir} (Drive disabled, or not every step synced yet)")

# Drop the model objects the smoke test still holds, then return the cache to the driver.
for _name in ("bundle", "generator"):
    _obj = globals().get(_name)
    if _obj is not None and hasattr(_obj, "model"):
        _obj.model = None
free_gpu()
print(json.dumps(FINAL, indent=2))
print("\nSESSION COMPLETE — nothing is left running. Now: Runtime -> Disconnect and delete runtime. "
      "An allocated-but-idle runtime bills Compute Units exactly like a busy one.")
